# Silver Layer

## Cleaning, Transformation & Validation

This notebook transforms the Bronze-layer Delta tables into clean, standardized, and trustworthy Silver-layer datasets using PySpark.

Steps:

- Load the Bronze Delta tables
- Inspect the Bronze schemas and columns
- Clean and standardize the data
- Standardize data types
- Validate and handle null values
- Identify and handle duplicate records
- Validate foreign key relationships
- Apply business rules and data quality validations
- Perform Silver-layer data quality checks
- Write the transformed data to Silver Delta tables
- Validate the Silver tables

### 2. Load Bronze Tables

The Bronze Delta tables are loaded from the Lakehouse into PySpark DataFrames for Silver-layer transformation and validation.

The Bronze layer remains unchanged. All cleaning, standardization, validation, and transformation operations are performed on DataFrames before the results are written to the Silver layer.

In [1]:
from  pyspark.sql import functions as F 

StatementMeta(, 60c04ed9-6876-4c6a-bfe0-1678ae3b54a8, 7, Finished, Available, Finished, False)

In [ ]:
df_bronze_customers = spark.table("1_bronze.bronze_customers")

df_bronze_order_items = spark.table("1_bronze.bronze_order_items")

df_bronze_orders = spark.table("1_bronze.bronze_orders")

df_bronze_payments = spark.table("1_bronze.bronze_payments")

df_bronze_pos_sales_transactions = spark.table(
    "1_bronze.bronze_pos_sales_transactions"
)

df_bronze_products = spark.table("1_bronze.bronze_products")

df_bronze_reviews = spark.table("1_bronze.bronze_reviews")

df_bronze_sellers = spark.table("1_bronze.bronze_sellers")

df_bronze_employees = spark.table("1_bronze.bronze_employees")

df_bronze_inventory_snapshots = spark.table(
    "1_bronze.bronze_inventory_snapshots"
)

df_bronze_promotions = spark.table("1_bronze.bronze_promotions")

df_bronze_stores = spark.table("1_bronze.bronze_stores")

df_bronze_suppliers = spark.table("1_bronze.bronze_suppliers")

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 15, Finished, Available, Finished, False)

### Bronze Table Load Verification

Verify that all Bronze tables have been successfully loaded into PySpark DataFrames before beginning Silver transformations.

In [ ]:
print("Bronze table load verification:")
print("-" * 40)

print("Customers:", df_bronze_customers.count())
print("Order Items:", df_bronze_order_items.count())
print("Orders:", df_bronze_orders.count())
print("Payments:", df_bronze_payments.count())
print("POS Sales Transactions:", df_bronze_pos_sales_transactions.count())
print("Products:", df_bronze_products.count())
print("Reviews:", df_bronze_reviews.count())
print("Sellers:", df_bronze_sellers.count())
print("Employees:", df_bronze_employees.count())
print("Inventory Snapshots:", df_bronze_inventory_snapshots.count())
print("Promotions:", df_bronze_promotions.count())
print("Stores:", df_bronze_stores.count())
print("Suppliers:", df_bronze_suppliers.count())

print("-" * 40)
print("All Bronze tables loaded successfully.")

StatementMeta(, f45c585c-3ffe-4ad1-8518-35084e05641d, 4, Finished, Available, Finished, False)

Bronze table load verification:
----------------------------------------
Customers: 10000
Order Items: 11335
Orders: 9946
Payments: 10356
POS Sales Transactions: 75000
Products: 6827
Reviews: 10362
Sellers: 1653
Employees: 150
Inventory Snapshots: 1500
Promotions: 30
Stores: 50
Suppliers: 250
----------------------------------------
All Bronze tables loaded successfully.


### 3. Bronze Schema Inspection

Inspect the schemas and columns of the Bronze DataFrames before applying Silver-layer transformations.

This step confirms the current column names and data types and helps identify fields that require standardization, type conversion, cleaning, or validation.

In [3]:
print("=" * 60)
print("CUSTOMERS")
print("=" * 60)
print(df_bronze_customers.columns)
df_bronze_customers.printSchema()


print("=" * 60)
print("ORDER ITEMS")
print("=" * 60)
print(df_bronze_order_items.columns)
df_bronze_order_items.printSchema()


print("=" * 60)
print("ORDERS")
print("=" * 60)
print(df_bronze_orders.columns)
df_bronze_orders.printSchema()


print("=" * 60)
print("PAYMENTS")
print("=" * 60)
print(df_bronze_payments.columns)
df_bronze_payments.printSchema()


print("=" * 60)
print("POS SALES TRANSACTIONS")
print("=" * 60)
print(df_bronze_pos_sales_transactions.columns)
df_bronze_pos_sales_transactions.printSchema()


print("=" * 60)
print("PRODUCTS")
print("=" * 60)
print(df_bronze_products.columns)
df_bronze_products.printSchema()


print("=" * 60)
print("REVIEWS")
print("=" * 60)
print(df_bronze_reviews.columns)
df_bronze_reviews.printSchema()


print("=" * 60)
print("SELLERS")
print("=" * 60)
print(df_bronze_sellers.columns)
df_bronze_sellers.printSchema()


print("=" * 60)
print("EMPLOYEES")
print("=" * 60)
print(df_bronze_employees.columns)
df_bronze_employees.printSchema()


print("=" * 60)
print("INVENTORY SNAPSHOTS")
print("=" * 60)
print(df_bronze_inventory_snapshots.columns)
df_bronze_inventory_snapshots.printSchema()


print("=" * 60)
print("PROMOTIONS")
print("=" * 60)
print(df_bronze_promotions.columns)
df_bronze_promotions.printSchema()


print("=" * 60)
print("STORES")
print("=" * 60)
print(df_bronze_stores.columns)
df_bronze_stores.printSchema()


print("=" * 60)
print("SUPPLIERS")
print("=" * 60)
print(df_bronze_suppliers.columns)
df_bronze_suppliers.printSchema()

StatementMeta(, f45c585c-3ffe-4ad1-8518-35084e05641d, 5, Finished, Available, Finished, False)

CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', '_source_file', '_ingestion_timestamp', '_ingestion_date']
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

ORDER ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', '_source_file', '_ingestion_timestamp', '_ingestion_date']
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = tru

#

### 4. Customers Dataset

In [1]:
from pyspark.sql import functions as F 

StatementMeta(, 4c76079e-c355-4bb9-94bb-29024d06c7df, 6, Finished, Available, Finished, False)

In [ ]:
df_bronze_customers = spark.table("1_bronze.bronze_customers")

StatementMeta(, 4c76079e-c355-4bb9-94bb-29024d06c7df, 8, Finished, Available, Finished, False)

#### 4.1.1 Null Validation

Check all columns in the Customers Bronze dataset for NULL values, including the Bronze ingestion metadata columns.

Result:

- No NULL values were detected across the Customers dataset.
- All columns passed the NULL validation check.

In [7]:
df_bronze_customers.select(
    [
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in df_bronze_customers.columns
    ]
).show()

StatementMeta(, c0f53270-3a47-4dbe-ae02-1d4a70c288a2, 9, Finished, Available, Finished, False)

+-----------+------------------+------------------------+-------------+--------------+------------+--------------------+---------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|_source_file|_ingestion_timestamp|_ingestion_date|
+-----------+------------------+------------------------+-------------+--------------+------------+--------------------+---------------+
|          0|                 0|                       0|            0|             0|           0|                   0|              0|
+-----------+------------------+------------------------+-------------+--------------+------------+--------------------+---------------+



#### 4.1.2 Blank / Whitespace String Validation

Check applicable string columns for empty or whitespace-only values that may represent missing data but are not stored as NULL.

Result:

- No blank or whitespace-only values were detected in the Customers string columns.
- All applicable string columns passed the blank / whitespace validation check.

In [8]:
string_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
]

for c in string_columns:
    blank_count = df_bronze_customers.filter(
        F.col(c).isNotNull() &
        (F.trim(F.col(c)) == "")
    ).count()

    print(f"{c}: {blank_count} blank/whitespace values")

StatementMeta(, c0f53270-3a47-4dbe-ae02-1d4a70c288a2, 10, Finished, Available, Finished, False)

customer_id: 0 blank/whitespace values
customer_unique_id: 0 blank/whitespace values
customer_city: 0 blank/whitespace values
customer_state: 0 blank/whitespace values


### 4.1.3 Duplicate customer_id validation

Check for duplicate `customer_id` values. Each customer record should have a unique `customer_id`.

In [9]:
duplicate_customer_ids = (
    df_bronze_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_customer_ids.show()

StatementMeta(, c0f53270-3a47-4dbe-ae02-1d4a70c288a2, 12, Finished, Available, Finished, False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



### 4.1.4. Data Type Validation

Inspect the Bronze DataFrame schema to verify that columns have appropriate data types before applying Silver-layer transformations.

In [12]:
df_bronze_customers.printSchema() 

StatementMeta(, c0f53270-3a47-4dbe-ae02-1d4a70c288a2, 21, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)



### 4.1.5. Invalid values (ZIP Code Validation)

Check the `customer_zip_code_prefix` column for negative values that would violate the expected numeric domain.

In [10]:
invalid_zip_codes = (
    df_bronze_customers
    .filter(F.col("customer_zip_code_prefix") < 0)
)

invalid_zip_codes.show()

StatementMeta(, c0f53270-3a47-4dbe-ae02-1d4a70c288a2, 15, Finished, Available, Finished, False)

+-----------+------------------+------------------------+-------------+--------------+------------+--------------------+---------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|_source_file|_ingestion_timestamp|_ingestion_date|
+-----------+------------------+------------------------+-------------+--------------+------------+--------------------+---------------+
+-----------+------------------+------------------------+-------------+--------------+------------+--------------------+---------------+



### 4.1.6. Data range (ZIP Code Range Validation)

Inspect the minimum and maximum `customer_zip_code_prefix` values to identify unexpected ranges before defining the Silver-layer validation rule.

In [11]:
zip_range = df_bronze_customers.select(
    F.min("customer_zip_code_prefix").alias("min_zip_code"),
    F.max("customer_zip_code_prefix").alias("max_zip_code")
)

zip_range.show()

StatementMeta(, c0f53270-3a47-4dbe-ae02-1d4a70c288a2, 19, Finished, Available, Finished, False)

+------------+------------+
|min_zip_code|max_zip_code|
+------------+------------+
|        1006|       99955|
+------------+------------+



### 4.1.7. Date / Timestamp Validation

The Customers dataset does not contain business date or timestamp columns.

The Bronze ingestion metadata columns `_ingestion_timestamp` and `_ingestion_date` are validated separately as technical audit metadata to ensure ingestion lineage and audit information is populated correctly.

### 4.1.8 . Foreign-Key Validation

Foreign-key validation is not applicable to the Customers dataset because `customer_id` is a parent key that is referenced by other datasets rather than a foreign key in this table.

In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


### 4.1.9. Business-Rule Validation

Validate customer attributes against expected business rules before defining the Silver-layer transformation rules.

Business rules:

- `customer_id` must not be NULL and should uniquely identify a customer.
- `customer_unique_id` must not be NULL or blank.
- `customer_city` must not be NULL or blank.
- `customer_state` must contain a valid state code.
- `customer_zip_code_prefix` must be a non-negative numeric value.

In [3]:
df_bronze_customers.select(
    "customer_state"
).distinct() \
 .orderBy("customer_state") \
 .show(100, truncate=False)

StatementMeta(, bb314808-6c51-4739-9f14-d6a97d6ddb79, 5, Finished, Available, Finished, False)

+--------------+
|customer_state|
+--------------+
|AC            |
|AL            |
|AM            |
|AP            |
|BA            |
|CE            |
|DF            |
|ES            |
|GO            |
|MA            |
|MG            |
|MS            |
|MT            |
|PA            |
|PB            |
|PE            |
|PI            |
|PR            |
|RJ            |
|RN            |
|RO            |
|RR            |
|RS            |
|SC            |
|SE            |
|SP            |
|TO            |
+--------------+



### 4.1.10 Define Customers Silver Transformation Rules

Based on the profiling results, the Customers dataset does not contain NULL values, blank or whitespace-only strings, duplicate customer IDs, negative ZIP-code values, or invalid customer state codes.

Silver-layer transformations will therefore focus on standardization while preserving valid source data.

Transformation rules:

- Trim leading and trailing whitespace from string attributes.
- Standardize `customer_zip_code_prefix` as an integer.
- Preserve the validated `customer_id` and `customer_unique_id` values.
- Preserve the validated customer state codes.
- Preserve Bronze ingestion metadata for data lineage and auditability.

### 4.1.11 Silver Data Quality Validation

Validate the Customers Silver DataFrame before persisting it as a Silver Delta table.

The validation confirms that the Silver DataFrame retains the expected columns, data types, row count, and data-quality rules established during Bronze profiling.

In [4]:
df_silver_customers = df_bronze_customers.select(
    F.trim(F.col("customer_id")).alias("customer_id"),
    F.trim(F.col("customer_unique_id")).alias("customer_unique_id"),
    F.col("customer_zip_code_prefix").cast("int").alias("customer_zip_code_prefix"),
    F.trim(F.col("customer_city")).alias("customer_city"),
    F.trim(F.col("customer_state")).alias("customer_state"),
    "_source_file",
    "_ingestion_timestamp",
    "_ingestion_date"
)

StatementMeta(, 4c76079e-c355-4bb9-94bb-29024d06c7df, 9, Finished, Available, Finished, False)

In [5]:
print("Silver Customers validation")
print("-" * 40)

print("Row count:", df_silver_customers.count())

print("\nColumns:")
print(df_silver_customers.columns)

print("\nSchema:")
df_silver_customers.printSchema()

StatementMeta(, 4c76079e-c355-4bb9-94bb-29024d06c7df, 10, Finished, Available, Finished, False)

Silver Customers validation
----------------------------------------
Row count: 10000

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', '_source_file', '_ingestion_timestamp', '_ingestion_date']

Schema:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)



In [6]:
print("Duplicate customer IDs:")

df_silver_customers.groupBy("customer_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

StatementMeta(, 4c76079e-c355-4bb9-94bb-29024d06c7df, 11, Finished, Available, Finished, False)

Duplicate customer IDs:
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [7]:
df_silver_customers.select(
    [
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in df_silver_customers.columns
    ]
).show()

StatementMeta(, 4c76079e-c355-4bb9-94bb-29024d06c7df, 12, Finished, Available, Finished, False)

+-----------+------------------+------------------------+-------------+--------------+------------+--------------------+---------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|_source_file|_ingestion_timestamp|_ingestion_date|
+-----------+------------------+------------------------+-------------+--------------+------------+--------------------+---------------+
|          0|                 0|                       0|            0|             0|           0|                   0|              0|
+-----------+------------------+------------------------+-------------+--------------+------------+--------------------+---------------+



### 4.1.12 Write Customers to Silver Delta Table

Persist the validated Customers Silver DataFrame as a Delta table in the Silver schema.

In [ ]:
df_silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_customers")

StatementMeta(, 4c76079e-c355-4bb9-94bb-29024d06c7df, 13, Finished, Available, Finished, False)

In [ ]:
# Verify Silver Customers after write
df_verify_customers = spark.table("2_silver.silver_customers")

print("=== SILVER CUSTOMERS POST-WRITE VERIFICATION ===")
print(f"Rows: {df_verify_customers.count()}")

df_verify_customers.printSchema()

df_verify_customers.show(10, truncate=False)

StatementMeta(, 4c76079e-c355-4bb9-94bb-29024d06c7df, 14, Finished, Available, Finished, False)

=== SILVER CUSTOMERS POST-WRITE VERIFICATION ===
Rows: 10000
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------------------------------+--------------------------------+------------------------+---------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+---------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city  |customer_state|_source_file                                                                     

In [10]:
print("=== CUSTOMER WHITESPACE CHECK ===")

print(
    "Customer IDs with leading/trailing whitespace:",
    df_verify_customers.filter(
        (F.col("customer_id") != F.trim(F.col("customer_id"))) |
        (F.col("customer_unique_id") != F.trim(F.col("customer_unique_id"))) |
        (F.col("customer_city") != F.trim(F.col("customer_city"))) |
        (F.col("customer_state") != F.trim(F.col("customer_state")))
    ).count()
)

StatementMeta(, 4c76079e-c355-4bb9-94bb-29024d06c7df, 15, Finished, Available, Finished, False)

=== CUSTOMER WHITESPACE CHECK ===
Customer IDs with leading/trailing whitespace: 0


### 5. Order_items Dataset

#### 5.1. Profiling 

In [6]:
from  pyspark.sql import functions as F 

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 12, Finished, Available, Finished, False)

In [ ]:
df_bronze_order_items = spark.table("1_bronze.bronze_order_items")

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 8, Finished, Available, Finished, False)

In [3]:

print("=== ORDER ITEMS: BASIC PROFILE ===")

print(f"Rows: {df_bronze_order_items.count()}")
print(f"Columns: {len(df_bronze_order_items.columns)}")

df_bronze_order_items.printSchema()

display(df_bronze_order_items.limit(10))

StatementMeta(, 60c04ed9-6876-4c6a-bfe0-1678ae3b54a8, 9, Finished, Available, Finished, False)

=== ORDER ITEMS: BASIC PROFILE ===
Rows: 11335
Columns: 10
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)



SynapseWidget(Synapse.DataFrame, fbe4c0e1-2cbd-4c8f-b531-95cdb23ff690)

#### 5.1.1 null validation 

In [4]:
print("=== NULL / BLANK PROFILE ===")

null_profile = df_bronze_order_items.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_bronze_order_items.columns
])

display(null_profile)

StatementMeta(, 60c04ed9-6876-4c6a-bfe0-1678ae3b54a8, 10, Finished, Available, Finished, False)

=== NULL / BLANK PROFILE ===


SynapseWidget(Synapse.DataFrame, 3f9660c8-1199-48c2-a589-83efdee9b0fa)

#### 5.1.2 Checking Duplicates 

In [5]:
print("=== DUPLICATE PROFILE ===")

total_rows = df_bronze_order_items.count()
distinct_rows = df_bronze_order_items.distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicate rows: {total_rows - distinct_rows}")

StatementMeta(, 60c04ed9-6876-4c6a-bfe0-1678ae3b54a8, 11, Finished, Available, Finished, False)

=== DUPLICATE PROFILE ===
Total rows: 11335
Distinct rows: 11335
Duplicate rows: 0


#### 5.1.4 Inspect the columns

In [6]:
print("=== COLUMN NAMES ===")

for c in df_bronze_order_items.columns:
    print(c)

StatementMeta(, 60c04ed9-6876-4c6a-bfe0-1678ae3b54a8, 12, Finished, Available, Finished, False)

=== COLUMN NAMES ===
order_id
order_item_id
product_id
seller_id
shipping_limit_date
price
freight_value
_source_file
_ingestion_timestamp
_ingestion_date


#### 5.1.5. Look at descriptive statistics

In [7]:
print("=== NUMERIC PROFILE ===")

display(df_bronze_order_items.describe())

StatementMeta(, 60c04ed9-6876-4c6a-bfe0-1678ae3b54a8, 13, Finished, Available, Finished, False)

=== NUMERIC PROFILE ===


SynapseWidget(Synapse.DataFrame, b7d164b8-377e-4189-9111-3a6fc673bec5)

#### 5.1.6 Check key business columns

In [8]:
display(
    df_bronze_order_items.select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ).limit(20)
)

StatementMeta(, 60c04ed9-6876-4c6a-bfe0-1678ae3b54a8, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 69dfa372-a704-4f33-80b7-aea361f44901)

In [3]:
print("=== ORDER ITEM COMPOSITE KEY PROFILE ===")

total_rows = df_bronze_order_items.count()

distinct_keys = (
    df_bronze_order_items
    .select("order_id", "order_item_id")
    .distinct()
    .count()
)

duplicate_keys = total_rows - distinct_keys

print(f"Total rows: {total_rows}")
print(f"Distinct (order_id, order_item_id): {distinct_keys}")
print(f"Duplicate composite keys: {duplicate_keys}")

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 9, Finished, Available, Finished, False)

=== ORDER ITEM COMPOSITE KEY PROFILE ===
Total rows: 11335
Distinct (order_id, order_item_id): 11335
Duplicate composite keys: 0


In [4]:
print("=== SHIPPING LIMIT DATE ===")

df_bronze_order_items.select(
    "shipping_limit_date"
).show(10, truncate=False)

df_bronze_order_items.select(
    "shipping_limit_date"
).printSchema()

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 10, Finished, Available, Finished, False)

=== SHIPPING LIMIT DATE ===
+-------------------+
|shipping_limit_date|
+-------------------+
|2018-07-31 17:30:39|
|2018-03-29 20:07:49|
|2017-11-30 06:30:55|
|2017-06-26 22:10:14|
|2017-11-29 23:13:38|
|2018-05-02 22:10:29|
|2017-12-12 01:29:45|
|2018-08-22 11:49:46|
|2018-07-10 07:31:33|
|2017-08-17 07:15:16|
+-------------------+
only showing top 10 rows

root
 |-- shipping_limit_date: timestamp (nullable = true)



In [7]:
display(
    df_bronze_order_items.select(
        F.min("shipping_limit_date").alias("min_shipping_limit_date"),
        F.max("shipping_limit_date").alias("max_shipping_limit_date")
    )
)

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8ac71cd4-9a67-4668-8c2e-6b9b34bcbf17)

In [8]:
print("=== ORDER ITEM BUSINESS RULES ===")

invalid_price = df_bronze_order_items.filter(
    F.col("price") < 0
).count()

invalid_freight = df_bronze_order_items.filter(
    F.col("freight_value") < 0
).count()

invalid_item_id = df_bronze_order_items.filter(
    F.col("order_item_id") <= 0
).count()

print(f"Negative prices: {invalid_price}")
print(f"Negative freight values: {invalid_freight}")
print(f"Invalid order_item_id values: {invalid_item_id}")

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 14, Finished, Available, Finished, False)

=== ORDER ITEM BUSINESS RULES ===
Negative prices: 0
Negative freight values: 0
Invalid order_item_id values: 0


In [10]:
print("=== ORDER_ID FOREIGN KEY VALIDATION ===")

invalid_orders = (
    df_bronze_order_items
    .select("order_id")
    .distinct()
    .join(
        df_bronze_orders.select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
)

print(f"Invalid order_id values: {invalid_orders.count()}")

display(invalid_orders.limit(10))

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 16, Finished, Available, Finished, False)

=== ORDER_ID FOREIGN KEY VALIDATION ===
Invalid order_id values: 0


SynapseWidget(Synapse.DataFrame, 11a35897-f409-4048-bae7-a1dd419df4da)

In [13]:
print("=== PRODUCT_ID FOREIGN KEY VALIDATION ===")

invalid_products = (
    df_bronze_order_items
    .select("product_id")
    .distinct()
    .join(
        df_bronze_products.select("product_id").distinct(),
        on="product_id",
        how="left_anti"
    )
)

print(f"Invalid product_id values: {invalid_products.count()}")

display(invalid_products.limit(10))

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 19, Finished, Available, Finished, False)

=== PRODUCT_ID FOREIGN KEY VALIDATION ===
Invalid product_id values: 0


SynapseWidget(Synapse.DataFrame, d60ae8c4-9634-4ce2-a6bf-2b67f7da00b5)

In [12]:
print("=== SELLER_ID FOREIGN KEY VALIDATION ===")

invalid_sellers = (
    df_bronze_order_items
    .select("seller_id")
    .distinct()
    .join(
        df_bronze_sellers.select("seller_id").distinct(),
        on="seller_id",
        how="left_anti"
    )
)

print(f"Invalid seller_id values: {invalid_sellers.count()}")

display(invalid_sellers.limit(10))

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 18, Finished, Available, Finished, False)

=== SELLER_ID FOREIGN KEY VALIDATION ===
Invalid seller_id values: 0


SynapseWidget(Synapse.DataFrame, 6b016a3b-d3fc-4bd0-bb68-7d25369dcb4b)

### Transformation Rules

- Trim string identifiers.
- Cast `order_item_id` to integer.
- Cast `price` and `freight_value` to `decimal(12,2)`.
- Preserve valid timestamp and ingestion metadata.
- No records removed because no data-quality violations were identified.

In [14]:
df_silver_order_items = (
    df_bronze_order_items
    .select(
        F.trim(F.col("order_id")).alias("order_id"),
        F.col("order_item_id").cast("int").alias("order_item_id"),
        F.trim(F.col("product_id")).alias("product_id"),
        F.trim(F.col("seller_id")).alias("seller_id"),
        F.col("shipping_limit_date").cast("timestamp").alias("shipping_limit_date"),
        F.col("price").cast("decimal(12,2)").alias("price"),
        F.col("freight_value").cast("decimal(12,2)").alias("freight_value"),
        F.col("_source_file"),
        F.col("_ingestion_timestamp"),
        F.col("_ingestion_date")
    )
)

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 20, Finished, Available, Finished, False)

In [15]:
print("=== SILVER ORDER ITEMS ===")

df_silver_order_items.printSchema()

display(df_silver_order_items.limit(10))

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 21, Finished, Available, Finished, False)

=== SILVER ORDER ITEMS ===
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)



SynapseWidget(Synapse.DataFrame, 2ddce077-6680-4cec-857f-ed344e8501a9)

In [16]:
print("=== SILVER ORDER ITEMS VALIDATION ===")

print(f"Bronze row count: {df_bronze_order_items.count()}")
print(f"Silver row count: {df_silver_order_items.count()}")

print("\n=== DUPLICATE COMPOSITE KEY CHECK ===")

duplicate_keys = (
    df_silver_order_items
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicate composite keys: {duplicate_keys.count()}")

print("\n=== BUSINESS RULE CHECK ===")

print(
    f"Negative prices: "
    f"{df_silver_order_items.filter(F.col('price') < 0).count()}"
)

print(
    f"Negative freight values: "
    f"{df_silver_order_items.filter(F.col('freight_value') < 0).count()}"
)

print(
    f"Invalid order_item_id: "
    f"{df_silver_order_items.filter(F.col('order_item_id') <= 0).count()}"
)

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 22, Finished, Available, Finished, False)

=== SILVER ORDER ITEMS VALIDATION ===
Bronze row count: 11335
Silver row count: 11335

=== DUPLICATE COMPOSITE KEY CHECK ===
Duplicate composite keys: 0

=== BUSINESS RULE CHECK ===
Negative prices: 0
Negative freight values: 0
Invalid order_item_id: 0


In [17]:
print("\n=== SILVER NULL CHECK ===")

display(
    df_silver_order_items.select([
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df_silver_order_items.columns
    ])
)

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 23, Finished, Available, Finished, False)


=== SILVER NULL CHECK ===


SynapseWidget(Synapse.DataFrame, 00f9d1d7-82c6-4fe9-96c1-1119e6a75215)

In [ ]:
df_silver_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_order_items")

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 24, Finished, Available, Finished, False)

In [ ]:
df_verify_order_items = spark.table("2_silver.silver_order_items")

print("=== SILVER ORDER ITEMS TABLE VERIFICATION ===")

print(f"Rows: {df_verify_order_items.count()}")

df_verify_order_items.printSchema()

display(df_verify_order_items.limit(10))

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 25, Finished, Available, Finished, False)

=== SILVER ORDER ITEMS TABLE VERIFICATION ===
Rows: 11335
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)



SynapseWidget(Synapse.DataFrame, 6a58dd54-2bc5-4825-9c16-82053bc0359a)

In [ ]:
spark.sql("SHOW TABLES IN 2_silver").filter(
    F.col("tableName") == "silver_order_items"
).show()

StatementMeta(, 41783dc5-184c-4a1f-be74-e4584f2d8c9e, 26, Finished, Available, Finished, False)

+--------------------+------------------+-----------+
|           namespace|         tableName|isTemporary|
+--------------------+------------------+-----------+
|`My workspace`.Re...|silver_order_items|      false|
+--------------------+------------------+-----------+



### 6. Orders_Dataset

#### BRONZE ORDERS — LOAD & INITIAL INSPECTION

In [ ]:
from pyspark.sql import functions as F

df_bronze_orders = spark.table("1_bronze.bronze_orders")

df_bronze_orders.printSchema()
df_bronze_orders.show(10, truncate=False)

print("Row count:", df_bronze_orders.count())

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 5, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+----

In [3]:
# === ORDERS PROFILING ===

print("=== NULL CHECK ===")

df_bronze_orders.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_orders.columns
]).show()


print("=== STATUS CHECK ===")

df_bronze_orders.groupBy("order_status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()


print("=== DUPLICATE CHECK ===")

df_bronze_orders.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== BLANK CHECK ===")

df_bronze_orders.select(
    F.sum(
        F.when(F.trim(F.col("order_id")) == "", 1).otherwise(0)
    ).alias("blank_order_id"),

    F.sum(
        F.when(F.trim(F.col("customer_id")) == "", 1).otherwise(0)
    ).alias("blank_customer_id"),

    F.sum(
        F.when(F.trim(F.col("order_status")) == "", 1).otherwise(0)
    ).alias("blank_order_status")
).show()

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 6, Finished, Available, Finished, False)

=== NULL CHECK ===
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+--------------------+---------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_source_file|_ingestion_timestamp|_ingestion_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+--------------------+---------------+
|       0|          0|           0|                       0|                1|                          96|                          204|                            0|           0|                   0|              0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+--

In [4]:
# === APPROVAL NULL CHECK ===

df_bronze_orders.filter(
    F.col("order_approved_at").isNull()
).show(truncate=False)

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 7, Finished, Available, Finished, False)

+--------------------------------+--------------------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_source_file                                                                                                                                                                   |_ingestion_timestamp      |_ingestion_date|
+--------------------------------+--------------------------------+------------+------------------------+-----------------+-------------

In [5]:
# === NULL STATUS CHECK ===

df_bronze_orders.filter(
    F.col("order_delivered_carrier_date").isNull()
).groupBy("order_status") \
 .count() \
 .orderBy(F.desc("count")) \
 .show()

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 8, Finished, Available, Finished, False)

+------------+-----+
|order_status|count|
+------------+-----+
|    canceled|   38|
|    invoiced|   33|
|  processing|   24|
|   delivered|    1|
+------------+-----+



In [6]:
# === DELIVERY NULL CHECK ===

df_bronze_orders.filter(
    F.col("order_delivered_customer_date").isNull()
).groupBy("order_status") \
 .count() \
 .orderBy(F.desc("count")) \
 .show()

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 9, Finished, Available, Finished, False)

+------------+-----+
|order_status|count|
+------------+-----+
|     shipped|  101|
|    canceled|   46|
|    invoiced|   33|
|  processing|   24|
+------------+-----+



In [7]:
# === TIMESTAMP ORDER CHECK ===

df_bronze_orders.filter(
    F.col("order_approved_at").isNotNull() &
    (F.col("order_approved_at") < F.col("order_purchase_timestamp"))
).show(truncate=False)

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 10, Finished, Available, Finished, False)

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+--------------------+---------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_source_file|_ingestion_timestamp|_ingestion_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+--------------------+---------------+
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+--------------------+---------------+



In [8]:
# === CARRIER DATE CHECK ===

df_bronze_orders.filter(
    F.col("order_delivered_carrier_date").isNotNull() &
    F.col("order_approved_at").isNotNull() &
    (F.col("order_delivered_carrier_date") < F.col("order_approved_at"))
).show(truncate=False)

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 11, Finished, Available, Finished, False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_source_file                                                                                                                                                                   |_ingestion_timestamp      |_ingestion_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+-------

In [9]:
# === CUSTOMER DELIVERY CHECK ===

df_bronze_orders.filter(
    F.col("order_delivered_customer_date").isNotNull() &
    F.col("order_delivered_carrier_date").isNotNull() &
    (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"))
).show(truncate=False)

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 12, Finished, Available, Finished, False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_source_file                                                                                                                                                                   |_ingestion_timestamp      |_ingestion_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+-------

In [10]:
# === ESTIMATED DELIVERY CHECK ===

df_bronze_orders.filter(
    F.col("order_delivered_customer_date").isNotNull() &
    (F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"))
).show(truncate=False)

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 13, Finished, Available, Finished, False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_source_file                                                                                                                                                                   |_ingestion_timestamp      |_ingestion_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+-------

In [11]:
# === TIMESTAMP VIOLATION COUNTS ===

print("=== APPROVAL VIOLATIONS ===")

print(
    df_bronze_orders.filter(
        F.col("order_approved_at").isNotNull() &
        (F.col("order_approved_at") < F.col("order_purchase_timestamp"))
    ).count()
)


print("=== CARRIER VIOLATIONS ===")

print(
    df_bronze_orders.filter(
        F.col("order_delivered_carrier_date").isNotNull() &
        F.col("order_approved_at").isNotNull() &
        (F.col("order_delivered_carrier_date") < F.col("order_approved_at"))
    ).count()
)


print("=== CUSTOMER DELIVERY VIOLATIONS ===")

print(
    df_bronze_orders.filter(
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("order_delivered_carrier_date").isNotNull() &
        (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"))
    ).count()
)

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 14, Finished, Available, Finished, False)

=== APPROVAL VIOLATIONS ===
0
=== CARRIER VIOLATIONS ===
135
=== CUSTOMER DELIVERY VIOLATIONS ===
4


In [12]:
# === CARRIER VIOLATION STATUS ===

df_bronze_orders.filter(
    F.col("order_delivered_carrier_date").isNotNull() &
    F.col("order_approved_at").isNotNull() &
    (F.col("order_delivered_carrier_date") < F.col("order_approved_at"))
).groupBy("order_status") \
 .count() \
 .orderBy(F.desc("count")) \
 .show()

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 15, Finished, Available, Finished, False)

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|  134|
|     shipped|    1|
+------------+-----+



In [13]:
# === DELIVERY VIOLATION STATUS ===

df_bronze_orders.filter(
    F.col("order_delivered_customer_date").isNotNull() &
    F.col("order_delivered_carrier_date").isNotNull() &
    (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"))
).groupBy("order_status") \
 .count() \
 .orderBy(F.desc("count")) \
 .show()

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 16, Finished, Available, Finished, False)

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|    4|
+------------+-----+



In [15]:
# === CUSTOMER FK CHECK ===

df_orphan_orders = (
    df_bronze_orders
    .join(
        df_bronze_customers.select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )
)

print("Orphan customer IDs:", df_orphan_orders.count())

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 18, Finished, Available, Finished, False)

Orphan customer IDs: 0


## Orders Profiling

* **9,946 rows** with appropriate existing data types.
* No NULLs or duplicates in `order_id`; no blank string values found.
* `order_status` contains five valid statuses.
* Delivery-related NULLs are mostly legitimate based on order status.
* **1 NULL** `order_approved_at` will be preserved because the correct value cannot be derived.
* **135 carrier** and **4 customer-delivery** timestamp anomalies were identified. These will be preserved rather than overwritten or removed.
* **0 orphan `customer_id` values** — foreign-key validation passed.

## Silver Transformation

The Orders table will undergo minimal transformation: standardize string fields, preserve valid data types and legitimate NULLs, retain ingestion metadata, remove duplicates if required, and preserve the original timestamp values. Final Silver validation will confirm the established data-quality and business rules.


In [16]:
# === ORDERS TRANSFORM ===

df_silver_orders = (
    df_bronze_orders

    # Standardize string columns
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("order_status", F.trim(F.col("order_status")))

    # Remove duplicate orders
    .dropDuplicates(["order_id"])
)

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 19, Finished, Available, Finished, False)

In [17]:
# === ORDERS TRANSFORM CHECK ===

df_silver_orders.printSchema()

df_silver_orders.show(10, truncate=False)

print("Silver Orders row count:", df_silver_orders.count())

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 20, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+----

In [18]:
# === ORDERS VALIDATION ===

print("=== NULL CHECK ===")

df_silver_orders.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_orders.columns
]).show()


print("=== DUPLICATE CHECK ===")

df_silver_orders.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== BLANK CHECK ===")

df_silver_orders.select(
    F.sum(
        F.when(F.trim(F.col("order_id")) == "", 1).otherwise(0)
    ).alias("blank_order_id"),

    F.sum(
        F.when(F.trim(F.col("customer_id")) == "", 1).otherwise(0)
    ).alias("blank_customer_id"),

    F.sum(
        F.when(F.trim(F.col("order_status")) == "", 1).otherwise(0)
    ).alias("blank_order_status")
).show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_orders.count()
silver_count = df_silver_orders.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Match:", bronze_count == silver_count)

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 21, Finished, Available, Finished, False)

=== NULL CHECK ===
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+--------------------+---------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_source_file|_ingestion_timestamp|_ingestion_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+--------------------+---------------+
|       0|          0|           0|                       0|                1|                          96|                          204|                            0|           0|                   0|              0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+--

#### Timestamp Validation

The Silver Orders validation confirmed the same results identified during profiling:

* **0** purchase-to-approval violations.
* **135** approval-to-carrier anomalies preserved from the source.
* **4** carrier-to-customer anomalies preserved from the source.

The anomalies were intentionally preserved because the correct timestamps cannot be reliably determined.


In [19]:
# === TIMESTAMP CHECK ===

print("=== APPROVAL VIOLATIONS ===")

print(
    df_silver_orders.filter(
        F.col("order_approved_at").isNotNull() &
        (F.col("order_approved_at") < F.col("order_purchase_timestamp"))
    ).count()
)


print("=== CARRIER VIOLATIONS ===")

print(
    df_silver_orders.filter(
        F.col("order_delivered_carrier_date").isNotNull() &
        F.col("order_approved_at").isNotNull() &
        (F.col("order_delivered_carrier_date") < F.col("order_approved_at"))
    ).count()
)


print("=== CUSTOMER DELIVERY VIOLATIONS ===")

print(
    df_silver_orders.filter(
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("order_delivered_carrier_date").isNotNull() &
        (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"))
    ).count()
)

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 22, Finished, Available, Finished, False)

=== APPROVAL VIOLATIONS ===
0
=== CARRIER VIOLATIONS ===
135
=== CUSTOMER DELIVERY VIOLATIONS ===
4


In [20]:
# === CUSTOMER FK CHECK ===

df_silver_orphan_orders = (
    df_silver_orders
    .join(
        df_bronze_customers.select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )
)

print("Orphan customer IDs:", df_silver_orphan_orders.count())

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 23, Finished, Available, Finished, False)

Orphan customer IDs: 0


#### Orders Validation

Silver Orders passed all core validation checks. Row count remained **9,946**, no duplicates or blank keys were found, the customer foreign-key check returned **0 orphans**, and identified timestamp anomalies were intentionally preserved.


In [ ]:
# === WRITE SILVER ORDERS ===

silver_orders_table = "2_silver.silver_orders"

df_silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_orders_table)

print(f"Silver table written successfully: {silver_orders_table}")

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 24, Finished, Available, Finished, False)

Silver table written successfully: silver.silver_orders


In [ ]:
spark.table("2_silver.silver_orders")

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 25, Finished, Available, Finished, False)

DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, _source_file: string, _ingestion_timestamp: timestamp, _ingestion_date: date]

In [ ]:
# === TABLE CHECK ===

df_silver_orders_check = spark.table("2_silver.silver_orders")

df_silver_orders_check.printSchema()

print("Silver Orders row count:", df_silver_orders_check.count())

df_silver_orders_check.show(10, truncate=False)

StatementMeta(, 021a826c-e195-4025-95da-c0ffd0336148, 26, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

Silver Orders row count: 9946
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-

### 7. Payments_Dataset

In [4]:
from pyspark.sql import functions as F

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 8, Finished, Available, Finished, False)

In [ ]:
# === PAYMENTS INSPECTION ===

df_bronze_payments = spark.table("1_bronze.bronze_payments")

df_bronze_payments.printSchema()

df_bronze_payments.show(10, truncate=False)

print("Row count:", df_bronze_payments.count())

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 6, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------------------------------+------------------+------------+--------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|_source_file                                                                                                                                                                    |_ingestion_timest

In [5]:
# === PAYMENTS PROFILING ===

print("=== NULL CHECK ===")

df_bronze_payments.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_payments.columns
]).show()


print("=== PAYMENT TYPE CHECK ===")

df_bronze_payments.groupBy("payment_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()


print("=== DUPLICATE CHECK ===")

df_bronze_payments.groupBy(
    "order_id",
    "payment_sequential"
).count() \
 .filter(F.col("count") > 1) \
 .show()


print("=== BLANK CHECK ===")

df_bronze_payments.select(
    F.sum(
        F.when(F.trim(F.col("order_id")) == "", 1).otherwise(0)
    ).alias("blank_order_id"),

    F.sum(
        F.when(F.trim(F.col("payment_type")) == "", 1).otherwise(0)
    ).alias("blank_payment_type")
).show()

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 9, Finished, Available, Finished, False)

=== NULL CHECK ===
+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|_source_file|_ingestion_timestamp|_ingestion_date|
+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+
|       0|                 0|           0|                   0|            0|           0|                   0|              0|
+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+

=== PAYMENT TYPE CHECK ===
+------------+-----+
|payment_type|count|
+------------+-----+
| credit_card| 7603|
|      boleto| 2019|
|     voucher|  552|
|  debit_card|  182|
+------------+-----+

=== DUPLICATE CHECK ===
+--------+------------------+-----+
|order_id|payment_sequential|count|
+--------+------------------+-----+
+--------+--

In [6]:
# === PAYMENT VALUE CHECK ===

df_bronze_payments.filter(
    F.col("payment_value") < 0
).show()

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 10, Finished, Available, Finished, False)

+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|_source_file|_ingestion_timestamp|_ingestion_date|
+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+
+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+



In [7]:
# === INSTALLMENT CHECK ===

df_bronze_payments.filter(
    F.col("payment_installments") <= 0
).show()

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 11, Finished, Available, Finished, False)

+--------------------+------------------+------------+--------------------+-------------+--------------------+--------------------+---------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|        _source_file|_ingestion_timestamp|_ingestion_date|
+--------------------+------------------+------------+--------------------+-------------+--------------------+--------------------+---------------+
|744bade1fcf9ff3f3...|                 2| credit_card|                   0|        58.69|abfss://58e58f37-...|2026-08-20 08:37:...|     2026-08-20|
+--------------------+------------------+------------+--------------------+-------------+--------------------+--------------------+---------------+



In [8]:
# === SEQUENTIAL CHECK ===

df_bronze_payments.filter(
    F.col("payment_sequential") <= 0
).show()

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 12, Finished, Available, Finished, False)

+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|_source_file|_ingestion_timestamp|_ingestion_date|
+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+
+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+



### Payment Rules

- No negative `payment_value` values were found.
- All `payment_sequential` values are positive.
- One record has `payment_installments = 0`, which violates the positive-installment rule.
- The invalid installment value will be preserved because the correct value cannot be reliably determined.

#### Perfect. Foreign-key validation passed.

In [ ]:
# === ORDER FK CHECK ===

df_silver_orders = spark.table("2_silver.silver_orders")

df_orphan_payments = (
    df_bronze_payments
    .join(
        df_silver_orders.select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
)

print("Orphan order IDs:", df_orphan_payments.count())
print("If the result is 0, every payment record references an existing order in silver_orders.")

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 16, Finished, Available, Finished, False)

Orphan order IDs: 0
If the result is 0, every payment record references an existing order in silver_orders.


### Payments Transformation

The Payments data is already well-structured. The Silver transformation will standardize string fields, enforce uniqueness on `(order_id, payment_sequential)`, preserve valid data types and source anomalies, and retain ingestion metadata.

In [13]:
# === PAYMENTS TRANSFORM ===

df_silver_payments = (
    df_bronze_payments
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("payment_type", F.trim(F.col("payment_type")))
    .dropDuplicates(["order_id", "payment_sequential"])
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 17, Finished, Available, Finished, False)

In [14]:
# === PAYMENTS TRANSFORM CHECK ===

df_silver_payments.printSchema()

df_silver_payments.show(10, truncate=False)

print("Silver Payments row count:", df_silver_payments.count())

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 18, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------------------------------+------------------+------------+--------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|_source_file                                                                                                                                                                    |_ingestion_timest

In [15]:
# === PAYMENTS VALIDATION ===

print("=== NULL CHECK ===")

df_silver_payments.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_payments.columns
]).show()


print("=== DUPLICATE CHECK ===")

df_silver_payments.groupBy(
    "order_id",
    "payment_sequential"
).count() \
 .filter(F.col("count") > 1) \
 .show()


print("=== BLANK CHECK ===")

df_silver_payments.select(
    F.sum(
        F.when(F.trim(F.col("order_id")) == "", 1).otherwise(0)
    ).alias("blank_order_id"),

    F.sum(
        F.when(F.trim(F.col("payment_type")) == "", 1).otherwise(0)
    ).alias("blank_payment_type")
).show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_payments.count()
silver_count = df_silver_payments.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Match:", bronze_count == silver_count)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 19, Finished, Available, Finished, False)

=== NULL CHECK ===
+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|_source_file|_ingestion_timestamp|_ingestion_date|
+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+
|       0|                 0|           0|                   0|            0|           0|                   0|              0|
+--------+------------------+------------+--------------------+-------------+------------+--------------------+---------------+

=== DUPLICATE CHECK ===
+--------+------------------+-----+
|order_id|payment_sequential|count|
+--------+------------------+-----+
+--------+------------------+-----+

=== BLANK CHECK ===
+--------------+------------------+
|blank_order_id|blank_payment_type|
+--------------+------------------+
|             0|                 0|
+------

In [16]:
# === PAYMENT RULE CHECK ===

print("=== PAYMENT VALUE CHECK ===")

print(
    "Negative payment values:",
    df_silver_payments.filter(
        F.col("payment_value") < 0
    ).count()
)


print("=== INSTALLMENT CHECK ===")

print(
    "Invalid installments:",
    df_silver_payments.filter(
        F.col("payment_installments") <= 0
    ).count()
)


print("=== SEQUENTIAL CHECK ===")

print(
    "Invalid sequential values:",
    df_silver_payments.filter(
        F.col("payment_sequential") <= 0
    ).count()
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 20, Finished, Available, Finished, False)

=== PAYMENT VALUE CHECK ===
Negative payment values: 0
=== INSTALLMENT CHECK ===
Invalid installments: 1
=== SEQUENTIAL CHECK ===
Invalid sequential values: 0


#### Payments Validation

Silver Payments passed the core validation checks. No NULLs, blanks, or duplicate payment keys were found, and the row count remained **10,356**. Payment value and sequential checks passed. One `payment_installments = 0` anomaly was preserved, and the foreign-key check returned **0 orphan order IDs**.

In [17]:
# === ORDER FK CHECK ===

df_silver_orphan_payments = (
    df_silver_payments
    .join(
        df_silver_orders.select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
)

print("Orphan order IDs:", df_silver_orphan_payments.count())
print("If the result is 0, every payment record references an existing order in silver_orders.")

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 21, Finished, Available, Finished, False)

Orphan order IDs: 0
If the result is 0, every payment record references an existing order in silver_orders.


In [ ]:
# === WRITE SILVER PAYMENTS ===

df_silver_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("2_silver.silver_payments")

print("Silver Payments table written successfully.")

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 22, Finished, Available, Finished, False)

Silver Payments table written successfully.


### 8. Pos sales transactions Dataset

In [20]:
from pyspark.sql import functions as F

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 24, Finished, Available, Finished, False)

In [ ]:
# === POS SALES INSPECTION ===

df_bronze_pos_sales_transactions = spark.table(
    "1_bronze.bronze_pos_sales_transactions"
)

df_bronze_pos_sales_transactions.printSchema()

df_bronze_pos_sales_transactions.show(10, truncate=False)

print("Row count:", df_bronze_pos_sales_transactions.count())

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 23, Finished, Available, Finished, False)

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- customer_ID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+-------+---------+--------------------------------+--------+-------------------+-----+-----------+--------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|Invoice|StockCode|Description                     |Quantity|InvoiceDate        |Price|customer_ID|Country       |_source_file                                             

### POS Sales Profiling

The POS Sales Transactions table contains **75,000 rows**.

- No NULLs were found in `Invoice`, `StockCode`, `Quantity`, `InvoiceDate`, `Price`, or `Country`.
- `Description` contains **295 NULLs** and `customer_ID` contains **16,983 NULLs**. These will be preserved because valid transaction information is still present.
- No blank string values were found in the business string columns.
 **230 duplicate-key groups** were identified, involving 460 rows.
- Further investigation found **161 exact duplicate groups**, involving 322 rows.
- Only exact duplicate rows will be removed during Silver transformation; records with legitimate differences such as quantity will be preserved.
- **1,685 negative quantities** were found, with **1,430 associated with `C` invoices**. These will be preserved as potential return/cancellation transactions.
- No zero quantities or negative prices were found.
- **423 zero-price transactions** were identified and will be preserved as potential special transactions.
- Invoice prefixes were identified as `5`, `4`, and `C`.

## Silver Transformation

The POS table will undergo minimal transformation after the remaining duplicate and business-rule checks are completed. Valid source information and legitimate transaction events will be preserved rather than replaced or removed.


In [21]:
# === POS SALES PROFILING ===

print("=== NULL CHECK ===")

df_bronze_pos_sales_transactions.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_pos_sales_transactions.columns
]).show()


print("=== BLANK CHECK ===")

df_bronze_pos_sales_transactions.select(
    F.sum(
        F.when(F.trim(F.col("Invoice")) == "", 1).otherwise(0)
    ).alias("blank_invoice"),

    F.sum(
        F.when(F.trim(F.col("StockCode")) == "", 1).otherwise(0)
    ).alias("blank_stock_code"),

    F.sum(
        F.when(F.trim(F.col("Description")) == "", 1).otherwise(0)
    ).alias("blank_description"),

    F.sum(
        F.when(F.trim(F.col("Country")) == "", 1).otherwise(0)
    ).alias("blank_country")
).show()


print("=== DUPLICATE CHECK ===")

df_bronze_pos_sales_transactions.groupBy(
    "Invoice",
    "StockCode",
    "InvoiceDate",
    "customer_ID"
).count() \
 .filter(F.col("count") > 1) \
 .show()


print("=== COUNTRY CHECK ===")

df_bronze_pos_sales_transactions.groupBy("Country") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 25, Finished, Available, Finished, False)

=== NULL CHECK ===
+-------+---------+-----------+--------+-----------+-----+-----------+-------+------------+--------------------+---------------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|customer_ID|Country|_source_file|_ingestion_timestamp|_ingestion_date|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+------------+--------------------+---------------+
|      0|        0|        295|       0|          0|    0|      16983|      0|           0|                   0|              0|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+------------+--------------------+---------------+

=== BLANK CHECK ===
+-------------+----------------+-----------------+-------------+
|blank_invoice|blank_stock_code|blank_description|blank_country|
+-------------+----------------+-----------------+-------------+
|            0|               0|                0|            0|
+-------------+----------------+-----------------+-----

In [22]:
# === DESCRIPTION NULL CHECK ===

df_bronze_pos_sales_transactions.filter(
    F.col("Description").isNull()
).select(
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "customer_ID",
    "Country"
).show(20, truncate=False) 

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 26, Finished, Available, Finished, False)

+-------+------------+-----------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode   |Description|Quantity|InvoiceDate        |Price|customer_ID|Country       |
+-------+------------+-----------+--------+-------------------+-----+-----------+--------------+
|497815 |22058       |NULL       |192     |2010-02-12 14:52:00|0.0  |NULL       |United Kingdom|
|552227 |84596J      |NULL       |70      |2011-05-06 15:27:00|0.0  |NULL       |United Kingdom|
|530883 |21770       |NULL       |-52     |2010-11-04 15:06:00|0.0  |NULL       |United Kingdom|
|513571 |47598       |NULL       |4       |2010-06-25 14:46:00|0.0  |NULL       |United Kingdom|
|503524 |48195       |NULL       |6       |2010-04-01 13:11:00|0.0  |NULL       |United Kingdom|
|530355 |84458       |NULL       |-3      |2010-11-02 15:51:00|0.0  |NULL       |United Kingdom|
|569927 |22171       |NULL       |9       |2011-10-06 17:45:00|0.0  |NULL       |United Kingdom|
|510922 |90102       |NULL    

In [23]:
# === CUSTOMER NULL CHECK ===

df_bronze_pos_sales_transactions.filter(
    F.col("customer_ID").isNull()
).select(
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Country"
).show(20, truncate=False)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 27, Finished, Available, Finished, False)

+-------+---------+--------------------------------+--------+-------------------+-----+--------------+
|Invoice|StockCode|Description                     |Quantity|InvoiceDate        |Price|Country       |
+-------+---------+--------------------------------+--------+-------------------+-----+--------------+
|540026 |21519    |GIN & TONIC DIET GREETING CARD  |2       |2011-01-04 13:25:00|0.85 |United Kingdom|
|506224 |84709B   |PINK OVAL JEWELLED MIRROR       |1       |2010-04-28 11:55:00|5.91 |United Kingdom|
|547788 |22845    |VINTAGE CREAM CAT FOOD CONTAINER|1       |2011-03-25 12:00:00|12.46|United Kingdom|
|538349 |22819    |BIRTHDAY CARD, RETRO SPOT       |2       |2010-12-10 14:59:00|0.85 |United Kingdom|
|562933 |21787    |RAIN PONCHO RETROSPOT           |3       |2011-08-10 16:51:00|1.63 |United Kingdom|
|572549 |22197    |POPCORN HOLDER                  |1       |2011-10-24 17:03:00|1.63 |United Kingdom|
|497815 |22058    |NULL                            |192     |2010-02-12 1

In [24]:
# === QUANTITY CHECK ===

print("Negative quantities:",
      df_bronze_pos_sales_transactions
      .filter(F.col("Quantity") < 0)
      .count())

print("Zero quantities:",
      df_bronze_pos_sales_transactions
      .filter(F.col("Quantity") == 0)
      .count())

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 28, Finished, Available, Finished, False)

Negative quantities: 1685
Zero quantities: 0


In [25]:
# === PRICE CHECK ===

print("Negative prices:",
      df_bronze_pos_sales_transactions
      .filter(F.col("Price") < 0)
      .count())

print("Zero prices:",
      df_bronze_pos_sales_transactions
      .filter(F.col("Price") == 0)
      .count())

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 29, Finished, Available, Finished, False)

Negative prices: 0
Zero prices: 423


In [26]:
# === INVOICE CHECK ===

df_bronze_pos_sales_transactions \
    .withColumn(
        "invoice_prefix",
        F.substring(F.col("Invoice"), 1, 1)
    ) \
    .groupBy("invoice_prefix") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 30, Finished, Available, Finished, False)

+--------------+-----+
|invoice_prefix|count|
+--------------+-----+
|             5|65992|
|             4| 7578|
|             C| 1430|
+--------------+-----+



In [27]:
# === NEGATIVE QUANTITY CHECK ===

df_bronze_pos_sales_transactions \
    .filter(F.col("Quantity") < 0) \
    .withColumn(
        "invoice_prefix",
        F.substring(F.col("Invoice"), 1, 1)
    ) \
    .groupBy("invoice_prefix") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 31, Finished, Available, Finished, False)

+--------------+-----+
|invoice_prefix|count|
+--------------+-----+
|             C| 1430|
|             5|  200|
|             4|   55|
+--------------+-----+



In [28]:
# === DUPLICATE COUNT ===

df_pos_duplicates = (
    df_bronze_pos_sales_transactions
    .groupBy(
        "Invoice",
        "StockCode",
        "InvoiceDate",
        "customer_ID"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate groups:", df_pos_duplicates.count())

print(
    "Rows involved in duplicate groups:",
    df_pos_duplicates
    .select(F.sum("count"))
    .collect()[0][0]
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 32, Finished, Available, Finished, False)

Duplicate groups: 230
Rows involved in duplicate groups: 460


In [29]:
# === DUPLICATE DETAILS ===

df_pos_duplicates.join(
    df_bronze_pos_sales_transactions,
    on=["Invoice", "StockCode", "InvoiceDate", "customer_ID"],
    how="inner"
).select(
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "customer_ID",
    "count"
).orderBy(
    "Invoice",
    "StockCode"
).show(30, truncate=False)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 33, Finished, Available, Finished, False)

+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+-----+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|customer_ID|count|
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+-----+
|490009 |22196    |SMALL HEART MEASURING SPOONS       |8       |2009-12-03 12:13:00|0.85 |12835.0    |2    |
|490009 |22196    |SMALL HEART MEASURING SPOONS       |15      |2009-12-03 12:13:00|0.85 |12835.0    |2    |
|491204 |21588    |RETRO SPOT GIANT  TUBE MATCHES     |1       |2009-12-10 13:59:00|2.55 |14648.0    |2    |
|491204 |21588    |RETRO SPOT GIANT  TUBE MATCHES     |1       |2009-12-10 13:59:00|2.55 |14648.0    |2    |
|491717 |21590    |KINGS CHOICE CIGAR BOX MATCHES     |3       |2009-12-13 15:54:00|1.25 |16725.0    |2    |
|491717 |21590    |KINGS CHOICE CIGAR BOX MATCHES     |1       |2009-12-13 15:54:00|1.25 |16725.0    |2    |
|491924 |84582    |

In [30]:
# === EXACT DUPLICATE CHECK ===

df_pos_exact_duplicates = (
    df_bronze_pos_sales_transactions
    .groupBy(
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "customer_ID",
        "Country"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Exact duplicate groups:", df_pos_exact_duplicates.count())

print(
    "Rows involved in exact duplicates:",
    df_pos_exact_duplicates
    .select(F.sum("count"))
    .collect()[0][0]
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 34, Finished, Available, Finished, False)

Exact duplicate groups: 161
Rows involved in exact duplicates: 322


### POS Sales Transformation

The POS data requires minimal transformation. Exact duplicate rows will be removed, while legitimate repeated transactions will be preserved. String fields will be trimmed, valid data types and NULL values will be preserved, and ingestion metadata will be retained.

In [31]:
# === POS SALES TRANSFORM ===

df_silver_pos_sales_transactions = (
    df_bronze_pos_sales_transactions

    # Standardize string columns
    .withColumn("Invoice", F.trim(F.col("Invoice")))
    .withColumn("StockCode", F.trim(F.col("StockCode")))
    .withColumn("Description", F.trim(F.col("Description")))
    .withColumn("Country", F.trim(F.col("Country")))

    # Remove only exact duplicate rows
    .dropDuplicates()
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 35, Finished, Available, Finished, False)

### POS Sales Validation

The Silver transformation removed **161 exact duplicate rows**, reducing the dataset from **75,000 to 74,839 rows**. Legitimate repeated transactions were preserved. The validation will confirm NULLs, blank values, remaining exact duplicates, and row-count reconciliation.

In [32]:
# === POS SALES TRANSFORM CHECK ===

df_silver_pos_sales_transactions.printSchema()

df_silver_pos_sales_transactions.show(10, truncate=False)

print(
    "Silver POS Sales row count:",
    df_silver_pos_sales_transactions.count()
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 36, Finished, Available, Finished, False)

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- customer_ID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|customer_ID|Country       |_source_file                                       

#### -`customer_ID` NULLs decreased from **16,983 to 16,954** because **29 exact duplicate rows containing NULL customer IDs were removed**. No NULL customer IDs were filled or modified.

In [33]:
# === SILVER POS VALIDATION ===

print("=== NULL CHECK ===")

df_silver_pos_sales_transactions.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_pos_sales_transactions.columns
]).show()


print("=== BLANK CHECK ===")

df_silver_pos_sales_transactions.select(
    F.sum(F.when(F.trim(F.col("Invoice")) == "", 1).otherwise(0)).alias("blank_invoice"),
    F.sum(F.when(F.trim(F.col("StockCode")) == "", 1).otherwise(0)).alias("blank_stock_code"),
    F.sum(F.when(F.trim(F.col("Description")) == "", 1).otherwise(0)).alias("blank_description"),
    F.sum(F.when(F.trim(F.col("Country")) == "", 1).otherwise(0)).alias("blank_country")
).show()


print("=== EXACT DUPLICATE CHECK ===")

df_silver_pos_sales_transactions.groupBy(
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "customer_ID",
    "Country"
).count() \
 .filter(F.col("count") > 1) \
 .show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_pos_sales_transactions.count()
silver_count = df_silver_pos_sales_transactions.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Rows removed:", bronze_count - silver_count)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 37, Finished, Available, Finished, False)

=== NULL CHECK ===
+-------+---------+-----------+--------+-----------+-----+-----------+-------+------------+--------------------+---------------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|customer_ID|Country|_source_file|_ingestion_timestamp|_ingestion_date|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+------------+--------------------+---------------+
|      0|        0|        295|       0|          0|    0|      16954|      0|           0|                   0|              0|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+------------+--------------------+---------------+

=== BLANK CHECK ===
+-------------+----------------+-----------------+-------------+
|blank_invoice|blank_stock_code|blank_description|blank_country|
+-------------+----------------+-----------------+-------------+
|            0|               0|                0|            0|
+-------------+----------------+-----------------+-----

### - Negative quantities decreased from **1,685 to 1,682** because **3 exact duplicate rows with negative quantities were removed**. Negative quantities were otherwise preserved as potential return/cancellation transactions.
- No zero quantities or negative prices were found.
- **423 zero-price transactions** were preserved as potential special transactions.

In [34]:
# === POS BUSINESS RULE CHECK ===

print("=== QUANTITY CHECK ===")

print(
    "Negative quantities:",
    df_silver_pos_sales_transactions
    .filter(F.col("Quantity") < 0)
    .count()
)

print(
    "Zero quantities:",
    df_silver_pos_sales_transactions
    .filter(F.col("Quantity") == 0)
    .count()
)


print("=== PRICE CHECK ===")

print(
    "Negative prices:",
    df_silver_pos_sales_transactions
    .filter(F.col("Price") < 0)
    .count()
)

print(
    "Zero prices:",
    df_silver_pos_sales_transactions
    .filter(F.col("Price") == 0)
    .count()
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 38, Finished, Available, Finished, False)

=== QUANTITY CHECK ===
Negative quantities: 1682
Zero quantities: 0
=== PRICE CHECK ===
Negative prices: 0
Zero prices: 423


In [ ]:
# === CUSTOMER FK CHECK ===

df_silver_customers = spark.table("2_silver.silver_customers")

df_pos_orphan_customers = (
    df_silver_pos_sales_transactions.alias("pos")
    .filter(F.col("pos.customer_ID").isNotNull())
    .join(
        df_silver_customers.select("customer_id").distinct().alias("cust"),
        F.col("pos.customer_ID").cast("long") ==
        F.col("cust.customer_id").cast("long"),
        how="left_anti"
    )
)

print("Orphan customer IDs:", df_pos_orphan_customers.count())

print(
    "If the result is 0, every non-NULL customer_ID "
    "references an existing customer in silver_customers."
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 40, Finished, Available, Finished, False)

Orphan customer IDs: 57885
If the result is 0, every non-NULL customer_ID references an existing customer in silver_customers.


In [37]:
# === CUSTOMER ID COMPARISON ===

print("POS customer_ID examples:")

df_silver_pos_sales_transactions \
    .select("customer_ID") \
    .filter(F.col("customer_ID").isNotNull()) \
    .show(10)


print("Silver customer_id examples:")

df_silver_customers \
    .select("customer_id") \
    .show(10, truncate=False)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 41, Finished, Available, Finished, False)

POS customer_ID examples:
+-----------+
|customer_ID|
+-----------+
|    15203.0|
|    17050.0|
|    13849.0|
|    12437.0|
|    15640.0|
|    16787.0|
|    16065.0|
|    16173.0|
|    13952.0|
|    17189.0|
+-----------+
only showing top 10 rows

Silver customer_id examples:
+--------------------------------+
|customer_id                     |
+--------------------------------+
|c7432c6d237ffd6aa36a007b4237ec38|
|7f399d641e2e2064470145178c9e8778|
|ba5642b730704dc0f74b7cf715b41ed5|
|0f346a2cc84ebb2d52f0759d0acfd030|
|d393b9491df482cf448e60aa9955b7f2|
|d9b4a26e122e830decf445a401ff0506|
|3d7ded9f88ad6b06bc3f16ef0987fc54|
|beabad0a90f659fbd73b571183bab1ec|
|c0a599a259226219c1ea0d3b8821ca54|
|6a8f01843537891ff93700664ca11c7b|
+--------------------------------+
only showing top 10 rows



#### POS FK Validation

Customer FK validation was investigated between POS `customer_ID` and Olist `customer_id`.

The two datasets come from different source systems and use different customer identifier domains, so no valid FK relationship exists between them. Customer FK validation was therefore excluded for the POS dataset.

In [ ]:
# === WRITE SILVER POS TABLE ===

df_silver_pos_sales_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_pos_sales_transactions")

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 42, Finished, Available, Finished, False)

In [ ]:
# === SILVER POS TABLE CHECK ===

df_silver_pos_check = spark.table(
    "2_silver.silver_pos_sales_transactions"
)

df_silver_pos_check.printSchema()

print(
    "Silver POS rows:",
    df_silver_pos_check.count()
)

df_silver_pos_check.show(10, truncate=False)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 46, Finished, Available, Finished, False)

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- customer_ID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

Silver POS rows: 74839
+-------+---------+---------------------------------+--------+-------------------+-----+-----------+--------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|Invoice|StockCode|Description                      |Quantity|InvoiceDate        |Price|customer_ID|Country       |_source_file                    

### 9. Products_Dataset 

## Products – Profiling & Silver Transformation

The Bronze Products table contains **6,827 records**.

### Profiling Findings
- `product_id` contains no NULLs or blank values.
- No duplicate `product_id` values were found.
- **99 records** contain NULL product attributes, mainly category, name length, description length, and photo quantity.
- These NULL values represent missing source data and will be preserved in Silver rather than replaced with fabricated values.
- **1 product has an invalid weight of `0.0 g`**; this will be converted to NULL.
- No invalid product name lengths, description lengths, photo quantities, or physical dimensions were found.
- Product categories are stored in Portuguese.

### Silver Transformations
- Corrected source column typos:
  - `product_name_lenght` → `product_name_length`
  - `product_description_lenght` → `product_description_length`
- Added `product_category_name_en` containing standardized English category names.
- Preserved the original Portuguese `product_category_name`.
- Converted the invalid `0.0 g` product weight to NULL.
- Preserved legitimate NULL values from the source.
- No rows were removed during transformation.

### Result
The Silver Products DataFrame contains **6,827 rows and 13 columns**.

### Product Foreign-Key Validation

- `silver_order_items.product_id` was validated against `silver_products.product_id`.
- Orphan product IDs: **0**
- Foreign-key validation **passed**.
- Every Order Item references an existing Product.


In [41]:
from pyspark.sql import functions as F

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 48, Finished, Available, Finished, False)

In [ ]:
df_bronze_products = spark.table(
    "1_bronze.bronze_products"
) 

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 49, Finished, Available, Finished, False)

In [43]:
# === BRONZE PRODUCTS PROFILING ===

df_bronze_products.printSchema()

df_bronze_products.show(10, truncate=False)

print("Row count:", df_bronze_products.count())

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 50, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: double (nullable = true)
 |-- product_description_lenght: double (nullable = true)
 |-- product_photos_qty: double (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+------

### Products Profiling

The Bronze Products table contains **6,827 records**.

Profiling will check NULLs, blank strings, duplicate product IDs, and numeric business rules for product dimensions, weight, photos, and text lengths. The product ID will also be validated against the related Brazilian e-commerce tables where an appropriate relationship exists.

In [44]:
# === NULL CHECK ===

df_bronze_products.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_products.columns
]).show()


# === BLANK CHECK ===

df_bronze_products.select(
    F.sum(
        F.when(F.trim(F.col("product_id")) == "", 1).otherwise(0)
    ).alias("blank_product_id"),

    F.sum(
        F.when(F.trim(F.col("product_category_name")) == "", 1).otherwise(0)
    ).alias("blank_product_category")
).show()


# === DUPLICATE CHECK ===

df_bronze_products.groupBy("product_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


# === PRODUCT CATEGORY CHECK ===

df_bronze_products.groupBy("product_category_name") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 51, Finished, Available, Finished, False)

+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+------------+--------------------+---------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|_source_file|_ingestion_timestamp|_ingestion_date|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+------------+--------------------+---------------+
|         0|                   99|                 99|                        99|                99|               1|                1|                1|               1|           0|                   0|              0|
+----------+---------------------+-------------------+--------------------------+------------------+----------------

In [45]:
# === NUMERIC CHECK ===

print("=== PRODUCT NAME LENGTH ===")

print(
    "Invalid values:",
    df_bronze_products
    .filter(F.col("product_name_lenght") < 0)
    .count()
)


print("=== DESCRIPTION LENGTH ===")

print(
    "Invalid values:",
    df_bronze_products
    .filter(F.col("product_description_lenght") < 0)
    .count()
)


print("=== PHOTOS CHECK ===")

print(
    "Invalid photo quantities:",
    df_bronze_products
    .filter(F.col("product_photos_qty") < 0)
    .count()
)


print("=== WEIGHT CHECK ===")

print(
    "Invalid weights:",
    df_bronze_products
    .filter(F.col("product_weight_g") <= 0)
    .count()
)


print("=== DIMENSIONS CHECK ===")

print(
    "Invalid length:",
    df_bronze_products
    .filter(F.col("product_length_cm") <= 0)
    .count()
)

print(
    "Invalid height:",
    df_bronze_products
    .filter(F.col("product_height_cm") <= 0)
    .count()
)

print(
    "Invalid width:",
    df_bronze_products
    .filter(F.col("product_width_cm") <= 0)
    .count()
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 52, Finished, Available, Finished, False)

=== PRODUCT NAME LENGTH ===
Invalid values: 0
=== DESCRIPTION LENGTH ===
Invalid values: 0
=== PHOTOS CHECK ===
Invalid photo quantities: 0
=== WEIGHT CHECK ===
Invalid weights: 1
=== DIMENSIONS CHECK ===
Invalid length: 0
Invalid height: 0
Invalid width: 0


### - **1 product has an invalid weight of `0.0 g`**. Product dimensions are valid, so the weight will be handled during Silver transformation.
- No invalid product name lengths, description lengths, photo quantities, or dimensions were found.

In [46]:
# === INVALID WEIGHT CHECK ===

df_bronze_products.filter(
    F.col("product_weight_g") <= 0
).show(20, truncate=False)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 53, Finished, Available, Finished, False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|_source_file                                                                                                                                                                    |_ingestion_timestamp      |_ingestion_date|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------

### NULL Product Attribute Analysis

The Products dataset contains 99 records with NULL product attributes.

The NULLs are primarily concentrated in:
- `product_category_name`
- `product_name_lenght`
- `product_description_lenght`
- `product_photos_qty`

Most affected records still contain valid product dimensions and weight, so the products remain identifiable through `product_id`.

One record contains NULLs across all product attributes.

**Decision:** Retain these records in the Silver layer and preserve the NULL values rather than introducing fabricated values. The `product_id` remains available for product identification and downstream relationships.

In [47]:
# === NULL PRODUCT RECORDS ===

df_bronze_products.filter(
    F.col("product_category_name").isNull() |
    F.col("product_name_lenght").isNull() |
    F.col("product_description_lenght").isNull() |
    F.col("product_photos_qty").isNull() |
    F.col("product_weight_g").isNull() |
    F.col("product_length_cm").isNull() |
    F.col("product_height_cm").isNull() |
    F.col("product_width_cm").isNull()
).show(100, truncate=False)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 54, Finished, Available, Finished, False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|_source_file                                                                                                                                                                    |_ingestion_timestamp      |_ingestion_date|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------

In [50]:
# === SILVER PRODUCTS TRANSFORMATION ===

df_silver_products = (
    df_bronze_products

    # Fix invalid zero weight
    .withColumn(
        "product_weight_g",
        F.when(
            F.col("product_weight_g") <= 0,
            None
        ).otherwise(F.col("product_weight_g"))
    )

    # Rename source column typos
    .withColumnRenamed(
        "product_name_lenght",
        "product_name_length"
    )
    .withColumnRenamed(
        "product_description_lenght",
        "product_description_length"
    )

    # Add English product category
    .withColumn(
        "product_category_name_en",
        F.when(F.col("product_category_name") == "cama_mesa_banho", "bed_bath_table")
         .when(F.col("product_category_name") == "esporte_lazer", "sports_leisure")
         .when(F.col("product_category_name") == "beleza_saude", "health_beauty")
         .when(F.col("product_category_name") == "moveis_decoracao", "furniture_decor")
         .when(F.col("product_category_name") == "utilidades_domesticas", "housewares")
         .when(F.col("product_category_name") == "informatica_acessorios", "computers_accessories")
         .when(F.col("product_category_name") == "automotivo", "automotive")
         .when(F.col("product_category_name") == "brinquedos", "toys")
         .when(F.col("product_category_name") == "relogios_presentes", "watches_gifts")
         .when(F.col("product_category_name") == "telefonia", "telephony")
         .when(F.col("product_category_name") == "bebes", "baby")
         .when(F.col("product_category_name") == "perfumaria", "perfumery")
         .when(F.col("product_category_name") == "cool_stuff", "cool_stuff")
         .when(F.col("product_category_name") == "papelaria", "stationery")
         .when(F.col("product_category_name") == "fashion_bolsas_e_acessorios", "fashion_bags_accessories")
         .when(F.col("product_category_name") == "pet_shop", "pet_shop")
         .when(F.col("product_category_name") == "ferramentas_jardim", "garden_tools")
         .when(F.col("product_category_name") == "eletronicos", "electronics")
         .otherwise(F.col("product_category_name"))
    )
)

df_silver_products.printSchema()

print(
    "Silver Products row count:",
    df_silver_products.count()
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 57, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: double (nullable = true)
 |-- product_description_length: double (nullable = true)
 |-- product_photos_qty: double (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)
 |-- product_category_name_en: string (nullable = true)

Silver Products row count: 6827


In [51]:
# === SILVER PRODUCTS VALIDATION ===

print("=== NULL CHECK ===")

df_silver_products.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_products.columns
]).show()


print("=== DUPLICATE PRODUCT ID CHECK ===")

df_silver_products.groupBy("product_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_products.count()
silver_count = df_silver_products.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Match:", bronze_count == silver_count)


print("=== INVALID WEIGHT CHECK ===")

print(
    "Invalid weights:",
    df_silver_products
    .filter(F.col("product_weight_g") <= 0)
    .count()
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 58, Finished, Available, Finished, False)

=== NULL CHECK ===
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+------------+--------------------+---------------+------------------------+
|product_id|product_category_name|product_name_length|product_description_length|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|_source_file|_ingestion_timestamp|_ingestion_date|product_category_name_en|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+------------+--------------------+---------------+------------------------+
|         0|                   99|                 99|                        99|                99|               2|                1|                1|               1|           0|                   0|              0|                      9

### Reload Silver Order Items for FK Validation

The Silver Order Items transformation was completed previously. Since the DataFrame is not currently available in the active notebook session, the persisted Silver table is loaded again.

This avoids repeating the transformation logic and allows us to perform the Product foreign-key validation using the existing Silver data.

**Relationship being validated:**

`silver_order_items.product_id → silver_products.product_id`

In [ ]:
df_silver_order_items = spark.table(
    "2_silver.silver_order_items"
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 60, Finished, Available, Finished, False)

In [54]:
# === PRODUCT FOREIGN KEY VALIDATION ===

df_orphan_order_items_products = (
    df_silver_order_items
    .join(
        df_silver_products.select("product_id").distinct(),
        on="product_id",
        how="left_anti"
    )
)

print(
    "Orphan product IDs:",
    df_orphan_order_items_products
    .select("product_id")
    .distinct()
    .count()
)

print(
    "If the result is 0, every order item references an existing product in silver_products."
)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 61, Finished, Available, Finished, False)

Orphan product IDs: 0
If the result is 0, every order item references an existing product in silver_products.


In [ ]:
# === WRITE SILVER PRODUCTS TABLE ===

df_silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_products")

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 62, Finished, Available, Finished, False)

In [ ]:
# === FINAL SILVER PRODUCTS CHECK ===

df_silver_products_check = spark.table(
    "2_silver.silver_products"
)

df_silver_products_check.printSchema()

print(
    "Silver Products row count:",
    df_silver_products_check.count()
)

df_silver_products_check.show(10, truncate=False)

StatementMeta(, 0b9b20a9-b72e-4155-bed0-0808b0a54748, 63, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: double (nullable = true)
 |-- product_description_length: double (nullable = true)
 |-- product_photos_qty: double (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)
 |-- product_category_name_en: string (nullable = true)

Silver Products row count: 6827
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------------------------------------------------------------------------------------------------------------

### 10. Reviews dataset

### Reviews — Bronze to Silver

The Bronze reviews table was loaded into a PySpark DataFrame and profiled for data quality issues.

### Load and Inspect
- Loaded `1_bronze.bronze_reviews` into `df_bronze_reviews`.
- Confirmed 10,362 rows and 10 columns.
- Inspected the schema and sample records.

### Profiling
- Checked NULL values across all columns.
- Checked duplicate `review_id` values.
- Reviewed the `review_score` distribution.
- Checked for empty strings.
- Checked the `review_creation_date` range.
- Generated a statistical profile.

### Data Quality Findings
- 430 invalid `review_id` values.
- 238 invalid `review_score` values.
- 7 invalid `review_creation_date` values.
- 436 records identified as structurally corrupted.
- Legitimate NULL values were retained where appropriate.
- `review_comment_title` and `review_comment_message` were treated as optional fields.

### Clean and Transform
- Removed structurally corrupted records.
- Preserved legitimate NULL values.
- Converted `review_score` from string to integer.
- Converted `review_creation_date` from string to timestamp.
- Trimmed review titles and messages.
- Converted empty/whitespace-only text values to NULL.

### Foreign Key Validation
- Validated `order_id` against `silver_orders`.
- Found 0 orphan order IDs.
- Confirmed valid non-null `order_id` values reference existing orders.

### Deduplication
- Identified 6 duplicate `review_id` values after cleaning.
- Removed duplicate review records.
- Reduced the Silver dataset from 9,926 to 9,920 rows.

### Next Step
- Perform final Silver validation.
- Write the validated DataFrame to `2_silver.silver_reviews`.


In [1]:
from pyspark.sql import functions as F

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 3, Finished, Available, Finished, False)

In [ ]:
df_bronze_reviews = spark.table("1_bronze.bronze_reviews") 


StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 4, Finished, Available, Finished, False)

In [ ]:
# === Load and inspect ===

df_bronze_reviews = spark.table("1_bronze.bronze_reviews")

df_bronze_reviews.printSchema()

df_bronze_reviews.show(10, truncate=False)

print(f"Rows: {df_bronze_reviews.count()}")
print(f"Columns: {len(df_bronze_reviews.columns)}")

StatementMeta(, d1a8943e-95e2-4d31-9a7e-7e593b453058, 5, Finished, Available, Finished, False)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+----------------------------------------+--------------------------------+------------+--------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [3]:
# === PROFILE ===

print("=== NULL CHECK ===")

df_bronze_reviews.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_reviews.columns
]).show()


print("=== DUPLICATE REVIEW ID CHECK ===")

df_bronze_reviews.groupBy("review_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== REVIEW SCORE DISTRIBUTION ===")

df_bronze_reviews.groupBy("review_score") \
    .count() \
    .orderBy("review_score") \
    .show()


print("=== EMPTY STRING CHECK ===")

print("Empty review IDs:",
      df_bronze_reviews.filter(F.col("review_id") == "").count())

print("Empty order IDs:",
      df_bronze_reviews.filter(F.col("order_id") == "").count())

print("Empty titles:",
      df_bronze_reviews.filter(F.col("review_comment_title") == "").count())

print("Empty messages:",
      df_bronze_reviews.filter(F.col("review_comment_message") == "").count())


print("=== DATE RANGE CHECK ===")

df_bronze_reviews.select(
    "review_creation_date"
).summary("min", "max").show()


print("=== STATISTICAL PROFILE ===")

df_bronze_reviews.describe().show(truncate=False)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 5, Finished, Available, Finished, False)

=== NULL CHECK ===
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+------------+--------------------+---------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|_source_file|_ingestion_timestamp|_ingestion_date|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+------------+--------------------+---------------+
|        0|     183|         192|                9174|                  6290|                 777|                    782|           0|                   0|              0|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+------------+--------------------+---------------+

=== DUPLICATE REVIEW ID CHECK ===
+--------------------+-----+
|           review_id|count|
+--------------------+-

In [4]:
# === MALFORMED REVIEW RECORDS ===

print("=== INVALID REVIEW SCORES ===")

df_bronze_reviews.filter(
    ~F.col("review_score").isin("1", "2", "3", "4", "5")
).select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(20, truncate=False)


print("=== MALFORMED REVIEW IDs ===")

df_bronze_reviews.filter(
    F.length("review_id") != 32
).select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(20, truncate=False)


print("=== MALFORMED CREATION DATES ===")

df_bronze_reviews.filter(
    F.to_timestamp("review_creation_date").isNull() &
    F.col("review_creation_date").isNotNull()
).select(
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
).show(20, truncate=False)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 6, Finished, Available, Finished, False)

=== INVALID REVIEW SCORES ===
+---------------------------------------------------------------------------+------------------------------------------------------------------------------------------+-------------------------------------+--------------------+----------------------+--------------------+-----------------------+
|review_id                                                                  |order_id                                                                                  |review_score                         |review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------------------------------------------------------------------------+------------------------------------------------------------------------------------------+-------------------------------------+--------------------+----------------------+--------------------+-----------------------+
|Recomendo."                                                                |2017-

In [5]:
# === DATA QUALITY CHECK ===

print("=== VALID REVIEW ID CHECK ===")

valid_review_id = F.col("review_id").rlike("^[a-fA-F0-9]{32}$")

print(
    "Invalid review IDs:",
    df_bronze_reviews.filter(~valid_review_id).count()
)


print("=== VALID REVIEW SCORE CHECK ===")

valid_review_score = F.col("review_score").isin("1", "2", "3", "4", "5")

print(
    "Invalid review scores:",
    df_bronze_reviews.filter(
        F.col("review_score").isNotNull() &
        ~valid_review_score
    ).count()
)


print("=== VALID CREATION DATE CHECK ===")

valid_creation_date = F.to_timestamp(
    F.col("review_creation_date")
).isNotNull()

print(
    "Invalid creation dates:",
    df_bronze_reviews.filter(
        F.col("review_creation_date").isNotNull() &
        ~valid_creation_date
    ).count()
)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 7, Finished, Available, Finished, False)

=== VALID REVIEW ID CHECK ===
Invalid review IDs: 430
=== VALID REVIEW SCORE CHECK ===
Invalid review scores: 238
=== VALID CREATION DATE CHECK ===
Invalid creation dates: 7


In [6]:
# === CORRUPTED RECORD CHECK ===

valid_review_id = F.col("review_id").rlike("^[a-fA-F0-9]{32}$")
valid_review_score = F.col("review_score").isin("1", "2", "3", "4", "5")
valid_creation_date = F.to_timestamp(
    F.col("review_creation_date")
).isNotNull()

df_corrupted_reviews = df_bronze_reviews.filter(
    ~valid_review_id |
    (
        F.col("review_score").isNotNull() &
        ~valid_review_score
    ) |
    (
        F.col("review_creation_date").isNotNull() &
        ~valid_creation_date
    )
)

print("Corrupted records:", df_corrupted_reviews.count())

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 8, Finished, Available, Finished, False)

Corrupted records: 436


In [11]:
# === CLEAN AND TRANSFORM ===

df_silver_reviews = (
    df_bronze_reviews
    .filter(valid_review_id)
    .filter(
        F.col("review_score").isNull() |
        valid_review_score
    )
    .filter(
        F.col("review_creation_date").isNull() |
        valid_creation_date
    )
    .withColumn(
        "review_score",
        F.col("review_score").cast("integer")
    )
    .withColumn(
        "review_creation_date",
        F.to_timestamp("review_creation_date")
    )
    .withColumn(
        "review_comment_title",
        F.when(
            F.trim(F.col("review_comment_title")) == "",
            None
        ).otherwise(
            F.trim(F.col("review_comment_title"))
        )
    )
    .withColumn(
        "review_comment_message",
        F.when(
            F.trim(F.col("review_comment_message")) == "",
            None
        ).otherwise(
            F.trim(F.col("review_comment_message"))
        )
    )
)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 13, Finished, Available, Finished, False)

In [ ]:
# === ORDER FOREIGN KEY VALIDATION ===

df_silver_orders = spark.table("2_silver.silver_orders")

df_orphan_reviews = (
    df_silver_reviews
    .filter(F.col("order_id").isNotNull())
    .join(
        df_silver_orders.select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
)

print("Orphan order IDs:", df_orphan_reviews.count())

df_orphan_reviews.select(
    "review_id",
    "order_id"
).show(10, truncate=False)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 14, Finished, Available, Finished, False)

Orphan order IDs: 0
+---------+--------+
|review_id|order_id|
+---------+--------+
+---------+--------+



In [13]:
# === DEDUPLICATION ===

print("Duplicate review IDs before deduplication:")

df_silver_reviews.groupBy("review_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

print("Rows before deduplication:", df_silver_reviews.count())

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 15, Finished, Available, Finished, False)

Duplicate review IDs before deduplication:
+--------------------+-----+
|           review_id|count|
+--------------------+-----+
|b86b60b19d7ff8f19...|    2|
|f6fca12a2dae92684...|    2|
|7f0a8cc1d2aa250b9...|    2|
|5f05184cbcf379ca2...|    2|
|ef4767983d3b56ce5...|    2|
|50159be3324818783...|    2|
+--------------------+-----+

Rows before deduplication: 9926


In [14]:
# === DEDUPLICATION ===

df_silver_reviews = df_silver_reviews.dropDuplicates(
    ["review_id"]
)

print("Rows after deduplication:", df_silver_reviews.count())

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 16, Finished, Available, Finished, False)

Rows after deduplication: 9920


In [15]:
# === SILVER REVIEWS VALIDATION ===

print("=== NULL CHECK ===")

df_silver_reviews.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_reviews.columns
]).show()


print("=== DUPLICATE REVIEW ID CHECK ===")

df_silver_reviews.groupBy("review_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_reviews.count()
silver_count = df_silver_reviews.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Corrupted rows removed:", bronze_count - 9926)
print("Duplicates removed:", 9926 - silver_count)


print("=== INVALID REVIEW SCORE CHECK ===")

print(
    "Invalid scores:",
    df_silver_reviews
    .filter(
        F.col("review_score").isNotNull() &
        ~F.col("review_score").isin(1, 2, 3, 4, 5)
    )
    .count()
)


print("=== INVALID REVIEW ID CHECK ===")

print(
    "Invalid review IDs:",
    df_silver_reviews
    .filter(
        ~F.col("review_id").rlike("^[a-fA-F0-9]{32}$")
    )
    .count()
)


print("=== ORDER FOREIGN KEY CHECK ===")

print(
    "Orphan order IDs:",
    df_silver_reviews
    .filter(F.col("order_id").isNotNull())
    .join(
        df_silver_orders.select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
    .count()
)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 17, Finished, Available, Finished, False)

=== NULL CHECK ===
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+------------+--------------------+---------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|_source_file|_ingestion_timestamp|_ingestion_date|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+------------+--------------------+---------------+
|        0|       0|           0|                8768|                  5866|                 351|                    351|           0|                   0|              0|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+------------+--------------------+---------------+

=== DUPLICATE REVIEW ID CHECK ===
+---------+-----+
|review_id|count|
+---------+-----+
+---------+-----+

=== ROW 

In [ ]:
# === WRITE SILVER REVIEWS ===

df_silver_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_reviews")

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 18, Finished, Available, Finished, False)

In [ ]:
# === VERIFY SILVER REVIEWS ===

df_silver_reviews_check = spark.table("2_silver.silver_reviews")

print("Rows:", df_silver_reviews_check.count())

df_silver_reviews_check.printSchema()

df_silver_reviews_check.show(10, truncate=False)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 19, Finished, Available, Finished, False)

Rows: 9920
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------------------------------+--------------------------------+------------+--------------------+----------------------+--------------------+-----------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|review_id                       |order_id                        |review_score|review_comment_title|rev

### 11. Sellers_Dataset

### Sellers — Bronze to Silver

The Bronze sellers table was loaded into a PySpark DataFrame and profiled for data quality issues.

### Load and Inspect
- Loaded `1_bronze.bronze_sellers` into `df_bronze_sellers`.
- Confirmed 1,653 rows and 7 columns.
- Inspected the schema and sample records.

### Profiling
- Checked NULL values across all columns.
- Checked duplicate `seller_id` values.
- Validated seller ID format.
- Validated ZIP code values.
- Reviewed seller state distribution.
- Checked for empty strings.
- Generated a statistical profile.

### Data Quality Findings
- No NULL values found.
- No duplicate `seller_id` values found.
- No invalid seller IDs found.
- No invalid ZIP codes found.
- No empty strings found.
- 20 distinct seller states identified.
- No structural corruption detected.

### Clean and Transform
- Trimmed seller city values.
- Standardized seller city capitalization.
- Standardized seller state codes to uppercase.
- No records were removed.

### Validation
- Bronze rows: 1,653.
- Silver rows: 1,653.
- Row count matched.
- No NULL values.
- No duplicate seller IDs.
- No invalid seller IDs.
- No invalid ZIP codes.
- No invalid state codes.

### Next Step
- Write the validated DataFrame to `2_silver.silver_sellers`.


In [20]:
from pyspark.sql import functions as F

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 22, Finished, Available, Finished, False)

In [ ]:
# === LOAD AND INSPECT ===

df_bronze_sellers = spark.table("1_bronze.bronze_sellers")

df_bronze_sellers.printSchema()

df_bronze_sellers.show(10, truncate=False)

print(f"Rows: {df_bronze_sellers.count()}")
print(f"Columns: {len(df_bronze_sellers.columns)}")

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 23, Finished, Available, Finished, False)

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------------------------------+----------------------+------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|seller_id                       |seller_zip_code_prefix|seller_city |seller_state|_source_file                                                                                                                                                                  |_ingestion_timestamp      |_ingestion_date|
+--------------------------------+----------------------+------

In [22]:
# === PROFILE ===

print("=== NULL CHECK ===")

df_bronze_sellers.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_sellers.columns
]).show()


print("=== DUPLICATE SELLER ID CHECK ===")

df_bronze_sellers.groupBy("seller_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== SELLER ID FORMAT CHECK ===")

print(
    "Invalid seller IDs:",
    df_bronze_sellers
    .filter(
        ~F.col("seller_id").rlike("^[a-fA-F0-9]{32}$")
    )
    .count()
)


print("=== ZIP CODE CHECK ===")

print(
    "Invalid ZIP codes:",
    df_bronze_sellers
    .filter(
        F.col("seller_zip_code_prefix").isNull() |
        (F.col("seller_zip_code_prefix") < 0)
    )
    .count()
)


print("=== STATE CHECK ===")

print("Distinct states:", df_bronze_sellers.select("seller_state").distinct().count())

df_bronze_sellers.groupBy("seller_state") \
    .count() \
    .orderBy("seller_state") \
    .show()


print("=== EMPTY STRING CHECK ===")

print(
    "Empty seller IDs:",
    df_bronze_sellers.filter(F.trim(F.col("seller_id")) == "").count()
)

print(
    "Empty seller cities:",
    df_bronze_sellers.filter(F.trim(F.col("seller_city")) == "").count()
)

print(
    "Empty seller states:",
    df_bronze_sellers.filter(F.trim(F.col("seller_state")) == "").count()
)


print("=== STATISTICAL PROFILE ===")

df_bronze_sellers.describe().show(truncate=False)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 24, Finished, Available, Finished, False)

=== NULL CHECK ===
+---------+----------------------+-----------+------------+------------+--------------------+---------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|_source_file|_ingestion_timestamp|_ingestion_date|
+---------+----------------------+-----------+------------+------------+--------------------+---------------+
|        0|                     0|          0|           0|           0|                   0|              0|
+---------+----------------------+-----------+------------+------------+--------------------+---------------+

=== DUPLICATE SELLER ID CHECK ===
+---------+-----+
|seller_id|count|
+---------+-----+
+---------+-----+

=== SELLER ID FORMAT CHECK ===
Invalid seller IDs: 0
=== ZIP CODE CHECK ===
Invalid ZIP codes: 0
=== STATE CHECK ===
Distinct states: 20
+------------+-----+
|seller_state|count|
+------------+-----+
|          AC|    1|
|          BA|    8|
|          CE|    4|
|          DF|   24|
|          ES|    7|
|          GO|   1

In [23]:
# === CLEAN AND TRANSFORM ===

df_silver_sellers = (
    df_bronze_sellers
    .withColumn(
        "seller_city",
        F.initcap(F.trim(F.col("seller_city")))
    )
    .withColumn(
        "seller_state",
        F.upper(F.trim(F.col("seller_state")))
    )
)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 25, Finished, Available, Finished, False)

In [24]:
# === SILVER SELLERS VALIDATION ===

print("=== NULL CHECK ===")

df_silver_sellers.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_sellers.columns
]).show()


print("=== DUPLICATE SELLER ID CHECK ===")

df_silver_sellers.groupBy("seller_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_sellers.count()
silver_count = df_silver_sellers.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Match:", bronze_count == silver_count)


print("=== INVALID SELLER ID CHECK ===")

print(
    "Invalid seller IDs:",
    df_silver_sellers
    .filter(
        ~F.col("seller_id").rlike("^[a-fA-F0-9]{32}$")
    )
    .count()
)


print("=== INVALID ZIP CODE CHECK ===")

print(
    "Invalid ZIP codes:",
    df_silver_sellers
    .filter(
        (F.col("seller_zip_code_prefix") < 0) |
        F.col("seller_zip_code_prefix").isNull()
    )
    .count()
)


print("=== STATE FORMAT CHECK ===")

print(
    "Invalid state codes:",
    df_silver_sellers
    .filter(
        ~F.col("seller_state").rlike("^[A-Z]{2}$")
    )
    .count()
)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 26, Finished, Available, Finished, False)

=== NULL CHECK ===
+---------+----------------------+-----------+------------+------------+--------------------+---------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|_source_file|_ingestion_timestamp|_ingestion_date|
+---------+----------------------+-----------+------------+------------+--------------------+---------------+
|        0|                     0|          0|           0|           0|                   0|              0|
+---------+----------------------+-----------+------------+------------+--------------------+---------------+

=== DUPLICATE SELLER ID CHECK ===
+---------+-----+
|seller_id|count|
+---------+-----+
+---------+-----+

=== ROW COUNT CHECK ===
Bronze rows: 1653
Silver rows: 1653
Match: True
=== INVALID SELLER ID CHECK ===
Invalid seller IDs: 0
=== INVALID ZIP CODE CHECK ===
Invalid ZIP codes: 0
=== STATE FORMAT CHECK ===
Invalid state codes: 0


In [ ]:
# === WRITE SILVER SELLERS ===

df_silver_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_sellers")

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 27, Finished, Available, Finished, False)

In [ ]:
# === VERIFY SILVER SELLERS ===

df_silver_sellers_check = spark.table("2_silver.silver_sellers")

print("Rows:", df_silver_sellers_check.count())

df_silver_sellers_check.printSchema()

df_silver_sellers_check.show(10, truncate=False)

StatementMeta(, 4a6b46a6-776e-473e-a63c-552d08b650af, 28, Finished, Available, Finished, False)

Rows: 1653
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------------------------------+----------------------+------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|seller_id                       |seller_zip_code_prefix|seller_city |seller_state|_source_file                                                                                                                                                                  |_ingestion_timestamp      |_ingestion_date|
+--------------------------------+------------------

### 12. Employees_Dataset

In [3]:
from pyspark.sql import functions as F 

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 5, Finished, Available, Finished, False)

### 12. Employees — Bronze to Silver

The Bronze employees table was loaded into a PySpark DataFrame and profiled for data quality issues.

### Load and Inspect
- Loaded `1_bronze.bronze_employees` into `df_bronze_employees`.
- Confirmed 150 rows and 10 columns.
- Inspected the schema and sample records.
- Confirmed the synthetic naming convention:
  - `Staff_1`, `Staff_2`, etc. represent employee first names.
  - `LN_1`, `LN_2`, etc. represent employee last names.

### Profiling
- Checked NULL values across all columns.
- Checked duplicate `employee_id` values.
- Validated employee ID format.
- Validated store ID format.
- Checked hire dates.
- Checked for empty strings.
- Reviewed department and job-title distributions.
- Generated a statistical profile.

### Data Quality Findings
- No NULL values found.
- No duplicate `employee_id` values found.
- No invalid employee IDs found.
- No invalid store IDs found.
- No missing hire dates found.
- No empty strings found.
- 4 departments identified.
- 4 job-title categories identified.
- No structural corruption detected.

### Clean and Transform
- Trimmed first and last names.
- Standardized first and last names to title case.
- Standardized department values.
- Standardized job titles.
- Standardized `store_id` values to uppercase.
- No records were removed.

### Validation
- Bronze rows: 150.
- Silver rows: 150.
- Row count matched.
- No NULL values.
- No duplicate employee IDs.
- No invalid employee IDs.
- No invalid store IDs.
- No invalid hire dates.

### Foreign Key Validation
- `store_id` foreign-key validation is deferred until `silver_stores` is created.
- The relationship will be validated against the completed Silver Stores table.

### Output
- Created and verified `silver.silver_employees`.
- Final Silver row count: 150.


In [ ]:
# === LOAD AND INSPECT ===

df_bronze_employees = spark.table("1_bronze.bronze_employees")

df_bronze_employees.printSchema()

df_bronze_employees.show(10, truncate=False)

print(f"Rows: {df_bronze_employees.count()}")
print(f"Columns: {len(df_bronze_employees.columns)}")

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 3, Finished, Available, Finished, False)

root
 |-- employee_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+-----------+----------+---------+----------+----------+--------+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|employee_id|first_name|last_name|department|job_title |store_id|hire_date |_source_file                                                                                                                                                                       |_inges

In [4]:
# === PROFILE ===

print("=== NULL CHECK ===")

df_bronze_employees.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_employees.columns
]).show()


print("=== DUPLICATE EMPLOYEE ID CHECK ===")

df_bronze_employees.groupBy("employee_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== EMPLOYEE ID FORMAT CHECK ===")

print(
    "Invalid employee IDs:",
    df_bronze_employees
    .filter(
        ~F.col("employee_id").rlike("^EMP_[0-9]{5}$")
    )
    .count()
)


print("=== STORE ID FORMAT CHECK ===")

print(
    "Invalid store IDs:",
    df_bronze_employees
    .filter(
        ~F.col("store_id").rlike("^STR_[0-9]{3}$")
    )
    .count()
)


print("=== HIRE DATE CHECK ===")

print(
    "Invalid hire dates:",
    df_bronze_employees
    .filter(F.col("hire_date").isNull())
    .count()
)


print("=== EMPTY STRING CHECK ===")

for c in ["first_name", "last_name", "department", "job_title", "store_id"]:
    print(
        f"Empty {c}:",
        df_bronze_employees.filter(
            F.trim(F.col(c)) == ""
        ).count()
    )


print("=== CATEGORY DISTRIBUTION ===")

print("Departments:")
df_bronze_employees.groupBy("department") \
    .count() \
    .orderBy("department") \
    .show()

print("Job titles:")
df_bronze_employees.groupBy("job_title") \
    .count() \
    .orderBy("job_title") \
    .show()


print("=== STATISTICAL PROFILE ===")

df_bronze_employees.describe().show(truncate=False)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 6, Finished, Available, Finished, False)

=== NULL CHECK ===
+-----------+----------+---------+----------+---------+--------+---------+------------+--------------------+---------------+
|employee_id|first_name|last_name|department|job_title|store_id|hire_date|_source_file|_ingestion_timestamp|_ingestion_date|
+-----------+----------+---------+----------+---------+--------+---------+------------+--------------------+---------------+
|          0|         0|        0|         0|        0|       0|        0|           0|                   0|              0|
+-----------+----------+---------+----------+---------+--------+---------+------------+--------------------+---------------+

=== DUPLICATE EMPLOYEE ID CHECK ===
+-----------+-----+
|employee_id|count|
+-----------+-----+
+-----------+-----+

=== EMPLOYEE ID FORMAT CHECK ===
Invalid employee IDs: 0
=== STORE ID FORMAT CHECK ===
Invalid store IDs: 0
=== HIRE DATE CHECK ===
Invalid hire dates: 0
=== EMPTY STRING CHECK ===
Empty first_name: 0
Empty last_name: 0
Empty department: 

In [5]:
# === CLEAN AND TRANSFORM ===

df_silver_employees = (
    df_bronze_employees
    .withColumn(
        "first_name",
        F.initcap(F.trim(F.col("first_name")))
    )
    .withColumn(
        "last_name",
        F.initcap(F.trim(F.col("last_name")))
    )
    .withColumn(
        "department",
        F.initcap(F.trim(F.col("department")))
    )
    .withColumn(
        "job_title",
        F.initcap(F.trim(F.col("job_title")))
    )
    .withColumn(
        "store_id",
        F.upper(F.trim(F.col("store_id")))
    )
)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 7, Finished, Available, Finished, False)

In [ ]:
# === EMPLOYEE STORE FOREIGN KEY CHECK ===

df_silver_employees = spark.table("2_silver.silver_employees")
df_silver_stores = spark.table("2_silver.silver_stores")

orphan_employee_stores = (
    df_silver_employees
    .join(
        df_silver_stores.select("store_id"),
        on="store_id",
        how="left_anti"
    )
)

print(
    "Orphan employee store IDs:",
    orphan_employee_stores.count()
)

orphan_employee_stores.select(
    "employee_id",
    "store_id"
).show()

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 32, Finished, Available, Finished, False)

Orphan employee store IDs: 0
+-----------+--------+
|employee_id|store_id|
+-----------+--------+
+-----------+--------+



In [6]:
# === SILVER EMPLOYEES VALIDATION ===

print("=== NULL CHECK ===")

df_silver_employees.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_employees.columns
]).show()


print("=== DUPLICATE EMPLOYEE ID CHECK ===")

df_silver_employees.groupBy("employee_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_employees.count()
silver_count = df_silver_employees.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Match:", bronze_count == silver_count)


print("=== INVALID EMPLOYEE ID CHECK ===")

print(
    "Invalid employee IDs:",
    df_silver_employees
    .filter(
        ~F.col("employee_id").rlike("^EMP_[0-9]{5}$")
    )
    .count()
)


print("=== INVALID STORE ID CHECK ===")

print(
    "Invalid store IDs:",
    df_silver_employees
    .filter(
        ~F.col("store_id").rlike("^STR_[0-9]{3}$")
    )
    .count()
)


print("=== HIRE DATE CHECK ===")

print(
    "Invalid hire dates:",
    df_silver_employees
    .filter(F.col("hire_date").isNull())
    .count()
)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 8, Finished, Available, Finished, False)

=== NULL CHECK ===
+-----------+----------+---------+----------+---------+--------+---------+------------+--------------------+---------------+
|employee_id|first_name|last_name|department|job_title|store_id|hire_date|_source_file|_ingestion_timestamp|_ingestion_date|
+-----------+----------+---------+----------+---------+--------+---------+------------+--------------------+---------------+
|          0|         0|        0|         0|        0|       0|        0|           0|                   0|              0|
+-----------+----------+---------+----------+---------+--------+---------+------------+--------------------+---------------+

=== DUPLICATE EMPLOYEE ID CHECK ===
+-----------+-----+
|employee_id|count|
+-----------+-----+
+-----------+-----+

=== ROW COUNT CHECK ===
Bronze rows: 150
Silver rows: 150
Match: True
=== INVALID EMPLOYEE ID CHECK ===
Invalid employee IDs: 0
=== INVALID STORE ID CHECK ===
Invalid store IDs: 0
=== HIRE DATE CHECK ===
Invalid hire dates: 0


In [ ]:
# === WRITE SILVER EMPLOYEES ===

df_silver_employees.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_employees")

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 9, Finished, Available, Finished, False)

In [ ]:
# === VERIFY SILVER EMPLOYEES ===

df_silver_employees_check = spark.table("2_silver.silver_employees")

print("Rows:", df_silver_employees_check.count())

df_silver_employees_check.printSchema()

df_silver_employees_check.show(10, truncate=False)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 10, Finished, Available, Finished, False)

Rows: 150
root
 |-- employee_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+-----------+----------+---------+----------+----------+--------+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|employee_id|first_name|last_name|department|job_title |store_id|hire_date |_source_file                                                                                                                                                                    

### 13. Inventory_Snapshots Dataset 

### Inventory Snapshots — Bronze to Silver

The Bronze inventory snapshots table was loaded into a PySpark DataFrame and profiled for data quality and business-rule consistency.

### Load and Inspect
- Loaded `1_bronze.bronze_inventory_snapshots` into `df_bronze_inventory_snapshots`.
- Confirmed 1,500 rows and 9 columns.
- Inspected the schema and sample records.

### Profiling
- Checked NULL values across all columns.
- Checked duplicate snapshot records using `snapshot_date`, `store_id`, and `product_id`.
- Validated store and product ID formats.
- Validated stock and safety-stock values.
- Reviewed stock-status distribution.
- Checked snapshot dates.
- Checked for empty strings.
- Generated a statistical profile.

### Data Quality Findings
- No NULL values found.
- No duplicate snapshot records found.
- No invalid store IDs found.
- No invalid product IDs found.
- No invalid stock values found.
- No empty strings found.
- Three snapshot dates identified:
  - 2026-08-08
  - 2026-08-09
  - 2026-08-10

### Business Rule Validation
- `stock_on_hand = 0` → `Out of Stock`
- `stock_on_hand < safety_stock_level` → `Reorder Alert`
- `stock_on_hand >= safety_stock_level` → `Healthy Bucket`
- No stock-status rule violations found.

### Clean and Transform
- Standardized `store_id` to uppercase.
- Trimmed `product_id`.
- Trimmed `stock_status`.
- No records were removed.

### Foreign Key Validation
- Validated `product_id` against `silver_products`.
- Found 0 orphan product IDs.
- `store_id` foreign-key validation is deferred until `silver_stores` is created.

### Next Step
- Perform final Silver validation.
- Write the validated DataFrame to `2_silver.silver_inventory_snapshots`.


In [ ]:
# === LOAD AND INSPECT ===

df_bronze_inventory_snapshots = spark.table(
    "1_bronze.bronze_inventory_snapshots"
)

df_bronze_inventory_snapshots.printSchema()

df_bronze_inventory_snapshots.show(10, truncate=False)

print(f"Rows: {df_bronze_inventory_snapshots.count()}")
print(f"Columns: {len(df_bronze_inventory_snapshots.columns)}")

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 11, Finished, Available, Finished, False)

root
 |-- snapshot_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_on_hand: integer (nullable = true)
 |-- safety_stock_level: integer (nullable = true)
 |-- stock_status: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+-------------+--------+--------------------------------+-------------+------------------+--------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|snapshot_date|store_id|product_id                      |stock_on_hand|safety_stock_level|stock_status  |_source_file                                                                                                                              

In [10]:
# === PROFILE ===

print("=== NULL CHECK ===")

df_bronze_inventory_snapshots.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_inventory_snapshots.columns
]).show()


print("=== DUPLICATE SNAPSHOT CHECK ===")

df_bronze_inventory_snapshots.groupBy(
    "snapshot_date",
    "store_id",
    "product_id"
) \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== STORE ID FORMAT CHECK ===")

print(
    "Invalid store IDs:",
    df_bronze_inventory_snapshots
    .filter(
        ~F.col("store_id").rlike("^STR_[0-9]{3}$")
    )
    .count()
)


print("=== PRODUCT ID FORMAT CHECK ===")

print(
    "Invalid product IDs:",
    df_bronze_inventory_snapshots
    .filter(
        ~F.col("product_id").rlike("^[a-fA-F0-9]{32}$")
    )
    .count()
)


print("=== STOCK VALUE CHECK ===")

print(
    "Invalid stock on hand:",
    df_bronze_inventory_snapshots
    .filter(
        F.col("stock_on_hand").isNull() |
        (F.col("stock_on_hand") < 0)
    )
    .count()
)

print(
    "Invalid safety stock:",
    df_bronze_inventory_snapshots
    .filter(
        F.col("safety_stock_level").isNull() |
        (F.col("safety_stock_level") < 0)
    )
    .count()
)


print("=== STOCK STATUS DISTRIBUTION ===")

df_bronze_inventory_snapshots.groupBy("stock_status") \
    .count() \
    .orderBy("stock_status") \
    .show()


print("=== DATE RANGE CHECK ===")

df_bronze_inventory_snapshots.select(
    "snapshot_date"
).summary("min", "max").show()


print("=== EMPTY STRING CHECK ===")

for c in ["store_id", "product_id", "stock_status"]:
    print(
        f"Empty {c}:",
        df_bronze_inventory_snapshots.filter(
            F.trim(F.col(c)) == ""
        ).count()
    )


print("=== STATISTICAL PROFILE ===")

df_bronze_inventory_snapshots.describe().show(truncate=False)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 12, Finished, Available, Finished, False)

=== NULL CHECK ===
+-------------+--------+----------+-------------+------------------+------------+------------+--------------------+---------------+
|snapshot_date|store_id|product_id|stock_on_hand|safety_stock_level|stock_status|_source_file|_ingestion_timestamp|_ingestion_date|
+-------------+--------+----------+-------------+------------------+------------+------------+--------------------+---------------+
|            0|       0|         0|            0|                 0|           0|           0|                   0|              0|
+-------------+--------+----------+-------------+------------------+------------+------------+--------------------+---------------+

=== DUPLICATE SNAPSHOT CHECK ===
+-------------+--------+----------+-----+
|snapshot_date|store_id|product_id|count|
+-------------+--------+----------+-----+
+-------------+--------+----------+-----+

=== STORE ID FORMAT CHECK ===
Invalid store IDs: 0
=== PRODUCT ID FORMAT CHECK ===
Invalid product IDs: 0
=== STOCK VA

In [11]:
# === SNAPSHOT DATE CHECK ===

print("Distinct snapshot dates:",
      df_bronze_inventory_snapshots
      .select("snapshot_date")
      .distinct()
      .count())

df_bronze_inventory_snapshots.select(
    "snapshot_date"
).distinct() \
    .orderBy("snapshot_date") \
    .show(20, truncate=False)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 13, Finished, Available, Finished, False)

Distinct snapshot dates: 3
+-------------+
|snapshot_date|
+-------------+
|2026-08-08   |
|2026-08-09   |
|2026-08-10   |
+-------------+



In [12]:
# === STOCK STATUS VALIDATION ===

print("=== STOCK STATUS BY STOCK LEVEL ===")

df_bronze_inventory_snapshots.groupBy(
    "stock_status"
).agg(
    F.min("stock_on_hand").alias("min_stock"),
    F.max("stock_on_hand").alias("max_stock"),
    F.avg("stock_on_hand").alias("avg_stock")
).show()


print("=== STATUS BUSINESS RULE CHECK ===")

print(
    "Out of Stock with stock > 0:",
    df_bronze_inventory_snapshots
    .filter(
        (F.col("stock_status") == "Out of Stock") &
        (F.col("stock_on_hand") > 0)
    )
    .count()
)

print(
    "Reorder Alert with stock >= safety stock:",
    df_bronze_inventory_snapshots
    .filter(
        (F.col("stock_status") == "Reorder Alert") &
        (F.col("stock_on_hand") >= F.col("safety_stock_level"))
    )
    .count()
)

print(
    "Healthy Bucket with stock < safety stock:",
    df_bronze_inventory_snapshots
    .filter(
        (F.col("stock_status") == "Healthy Bucket") &
        (F.col("stock_on_hand") < F.col("safety_stock_level"))
    )
    .count()
)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 14, Finished, Available, Finished, False)

=== STOCK STATUS BY STOCK LEVEL ===
+--------------+---------+---------+-----------------+
|  stock_status|min_stock|max_stock|        avg_stock|
+--------------+---------+---------+-----------------+
| Reorder Alert|        1|       14|8.669421487603305|
|Healthy Bucket|       15|      149|81.45461200585652|
|  Out of Stock|        0|        0|              0.0|
+--------------+---------+---------+-----------------+

=== STATUS BUSINESS RULE CHECK ===
Out of Stock with stock > 0: 0
Reorder Alert with stock >= safety stock: 0
Healthy Bucket with stock < safety stock: 0


In [13]:
# === CLEAN AND TRANSFORM ===

df_silver_inventory_snapshots = (
    df_bronze_inventory_snapshots
    .withColumn(
        "store_id",
        F.upper(F.trim(F.col("store_id")))
    )
    .withColumn(
        "product_id",
        F.trim(F.col("product_id"))
    )
    .withColumn(
        "stock_status",
        F.trim(F.col("stock_status"))
    )
)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 15, Finished, Available, Finished, False)

In [ ]:
# === INVENTORY SNAPSHOT → STORE FOREIGN KEY CHECK ===

df_silver_inventory_snapshots = spark.table(
    "2_silver.silver_inventory_snapshots"
)

df_silver_stores = spark.table(
    "2_silver.silver_stores"
)

orphan_inventory_stores = (
    df_silver_inventory_snapshots
    .join(
        df_silver_stores.select("store_id"),
        on="store_id",
        how="left_anti"
    )
)

print(
    "Orphan inventory store IDs:",
    orphan_inventory_stores.count()
)

orphan_inventory_stores.select(
    "snapshot_date",
    "store_id",
    "product_id"
).show()

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 33, Finished, Available, Finished, False)

Orphan inventory store IDs: 0
+-------------+--------+----------+
|snapshot_date|store_id|product_id|
+-------------+--------+----------+
+-------------+--------+----------+



In [ ]:
# === PRODUCT FOREIGN KEY VALIDATION ===

df_silver_products = spark.table("2_silver.silver_products")

df_orphan_inventory_products = (
    df_silver_inventory_snapshots
    .filter(F.col("product_id").isNotNull())
    .join(
        df_silver_products.select("product_id").distinct(),
        on="product_id",
        how="left_anti"
    )
)

print(
    "Orphan product IDs:",
    df_orphan_inventory_products.count()
)

df_orphan_inventory_products.select(
    "snapshot_date",
    "store_id",
    "product_id"
).show(10, truncate=False)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 16, Finished, Available, Finished, False)

Orphan product IDs: 0
+-------------+--------+----------+
|snapshot_date|store_id|product_id|
+-------------+--------+----------+
+-------------+--------+----------+



In [15]:
# === SILVER INVENTORY SNAPSHOTS VALIDATION ===

print("=== NULL CHECK ===")

df_silver_inventory_snapshots.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_inventory_snapshots.columns
]).show()


print("=== DUPLICATE SNAPSHOT CHECK ===")

df_silver_inventory_snapshots.groupBy(
    "snapshot_date",
    "store_id",
    "product_id"
).count() \
 .filter(F.col("count") > 1) \
 .show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_inventory_snapshots.count()
silver_count = df_silver_inventory_snapshots.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Match:", bronze_count == silver_count)


print("=== INVALID PRODUCT ID CHECK ===")

print(
    "Invalid product IDs:",
    df_silver_inventory_snapshots
    .filter(
        ~F.col("product_id").rlike("^[a-fA-F0-9]{32}$")
    )
    .count()
)


print("=== INVALID STORE ID CHECK ===")

print(
    "Invalid store IDs:",
    df_silver_inventory_snapshots
    .filter(
        ~F.col("store_id").rlike("^STR_[0-9]{3}$")
    )
    .count()
)


print("=== STOCK BUSINESS RULE CHECK ===")

print(
    "Negative stock values:",
    df_silver_inventory_snapshots
    .filter(F.col("stock_on_hand") < 0)
    .count()
)

print(
    "Negative safety stock values:",
    df_silver_inventory_snapshots
    .filter(F.col("safety_stock_level") < 0)
    .count()
)

print(
    "Invalid stock status rules:",
    df_silver_inventory_snapshots
    .filter(
        ((F.col("stock_status") == "Out of Stock") &
         (F.col("stock_on_hand") > 0)) |
        ((F.col("stock_status") == "Reorder Alert") &
         (F.col("stock_on_hand") >= F.col("safety_stock_level"))) |
        ((F.col("stock_status") == "Healthy Bucket") &
         (F.col("stock_on_hand") < F.col("safety_stock_level")))
    )
    .count()
)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 17, Finished, Available, Finished, False)

=== NULL CHECK ===
+-------------+--------+----------+-------------+------------------+------------+------------+--------------------+---------------+
|snapshot_date|store_id|product_id|stock_on_hand|safety_stock_level|stock_status|_source_file|_ingestion_timestamp|_ingestion_date|
+-------------+--------+----------+-------------+------------------+------------+------------+--------------------+---------------+
|            0|       0|         0|            0|                 0|           0|           0|                   0|              0|
+-------------+--------+----------+-------------+------------------+------------+------------+--------------------+---------------+

=== DUPLICATE SNAPSHOT CHECK ===
+-------------+--------+----------+-----+
|snapshot_date|store_id|product_id|count|
+-------------+--------+----------+-----+
+-------------+--------+----------+-----+

=== ROW COUNT CHECK ===
Bronze rows: 1500
Silver rows: 1500
Match: True
=== INVALID PRODUCT ID CHECK ===
Invalid produ

In [ ]:
# === WRITE SILVER INVENTORY SNAPSHOTS ===

df_silver_inventory_snapshots.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_inventory_snapshots")

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 18, Finished, Available, Finished, False)

In [ ]:
# === VERIFY SILVER INVENTORY SNAPSHOTS ===

df_silver_inventory_snapshots_check = spark.table(
    "2_silver.silver_inventory_snapshots"
)

print("Rows:", df_silver_inventory_snapshots_check.count())

df_silver_inventory_snapshots_check.printSchema()

df_silver_inventory_snapshots_check.show(10, truncate=False)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 19, Finished, Available, Finished, False)

Rows: 1500
root
 |-- snapshot_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_on_hand: integer (nullable = true)
 |-- safety_stock_level: integer (nullable = true)
 |-- stock_status: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+-------------+--------+--------------------------------+-------------+------------------+--------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|snapshot_date|store_id|product_id                      |stock_on_hand|safety_stock_level|stock_status  |_source_file                                                                                                                   

### 14. Promotions_Dataset

### Promotions — Bronze to Silver

The Bronze promotions table was loaded into a PySpark DataFrame and profiled for data quality and business-rule consistency.

### Load and Inspect
- Loaded `1_bronze.bronze_promotions` into `df_bronze_promotions`.
- Confirmed 30 rows and 8 columns.
- Inspected the schema and sample records.

### Profiling
- Checked NULL values across all columns.
- Checked duplicate `promotion_id` values.
- Validated promotion ID format.
- Validated discount percentage values.
- Reviewed marketing channel distribution.
- Reviewed stackable promotion distribution.
- Checked for empty strings.
- Generated a statistical profile.

### Data Quality Findings
- No NULL values found.
- No duplicate `promotion_id` values found.
- No invalid promotion IDs found.
- No invalid discount values found.
- No empty strings found.
- Discount range: 5%–30%.
- 4 marketing channels identified.
- 3 stackable promotions and 27 non-stackable promotions.

### Clean and Transform
- Trimmed campaign names.
- Standardized promotion IDs to uppercase.
- Trimmed marketing channel values.
- No records were removed.

### Business Rule Validation
- `discount_percentage` values are within the valid range of 0–1.
- No invalid discount values found.

### Validation
- Bronze rows: 30.
- Silver rows: 30.
- Row count matched.
- No NULL values.
- No duplicate promotion IDs.
- No invalid promotion IDs.
- No invalid discounts.
- No empty strings.

### Output
- Ready to write the validated DataFrame to `silver.silver_promotions`.


In [ ]:
# === LOAD AND INSPECT ===

df_bronze_promotions = spark.table("1_bronze.bronze_promotions")

df_bronze_promotions.printSchema()

df_bronze_promotions.show(10, truncate=False)

print(f"Rows: {df_bronze_promotions.count()}")
print(f"Columns: {len(df_bronze_promotions.columns)}")

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 20, Finished, Available, Finished, False)

root
 |-- promotion_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- is_stackable: boolean (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+------------+----------------------+-------------------+-----------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|promotion_id|campaign_name         |discount_percentage|marketing_channel|is_stackable|_source_file                                                                                                                                                                        |_ingestion_timestamp      |_ingest

In [19]:
# === PROFILE ===

print("=== NULL CHECK ===")

df_bronze_promotions.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_promotions.columns
]).show()


print("=== DUPLICATE PROMOTION ID CHECK ===")

df_bronze_promotions.groupBy("promotion_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== PROMOTION ID FORMAT CHECK ===")

print(
    "Invalid promotion IDs:",
    df_bronze_promotions
    .filter(
        ~F.col("promotion_id").rlike("^PRM_[0-9]{3}$")
    )
    .count()
)


print("=== DISCOUNT CHECK ===")

print(
    "Invalid discounts:",
    df_bronze_promotions
    .filter(
        F.col("discount_percentage").isNull() |
        (F.col("discount_percentage") < 0) |
        (F.col("discount_percentage") > 1)
    )
    .count()
)


print("=== MARKETING CHANNEL DISTRIBUTION ===")

df_bronze_promotions.groupBy("marketing_channel") \
    .count() \
    .orderBy("marketing_channel") \
    .show()


print("=== STACKABLE DISTRIBUTION ===")

df_bronze_promotions.groupBy("is_stackable") \
    .count() \
    .show()


print("=== EMPTY STRING CHECK ===")

for c in ["promotion_id", "campaign_name", "marketing_channel"]:
    print(
        f"Empty {c}:",
        df_bronze_promotions.filter(
            F.trim(F.col(c)) == ""
        ).count()
    )


print("=== STATISTICAL PROFILE ===")

df_bronze_promotions.describe().show(truncate=False)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 21, Finished, Available, Finished, False)

=== NULL CHECK ===
+------------+-------------+-------------------+-----------------+------------+------------+--------------------+---------------+
|promotion_id|campaign_name|discount_percentage|marketing_channel|is_stackable|_source_file|_ingestion_timestamp|_ingestion_date|
+------------+-------------+-------------------+-----------------+------------+------------+--------------------+---------------+
|           0|            0|                  0|                0|           0|           0|                   0|              0|
+------------+-------------+-------------------+-----------------+------------+------------+--------------------+---------------+

=== DUPLICATE PROMOTION ID CHECK ===
+------------+-----+
|promotion_id|count|
+------------+-----+
+------------+-----+

=== PROMOTION ID FORMAT CHECK ===
Invalid promotion IDs: 0
=== DISCOUNT CHECK ===
Invalid discounts: 0
=== MARKETING CHANNEL DISTRIBUTION ===
+-----------------+-----+
|marketing_channel|count|
+-------------

In [20]:
# === CLEAN AND TRANSFORM ===

df_silver_promotions = (
    df_bronze_promotions
    .withColumn(
        "promotion_id",
        F.upper(F.trim(F.col("promotion_id")))
    )
    .withColumn(
        "campaign_name",
        F.trim(F.col("campaign_name"))
    )
    .withColumn(
        "marketing_channel",
        F.trim(F.col("marketing_channel"))
    )
)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 22, Finished, Available, Finished, False)

In [21]:
# === SILVER PROMOTIONS VALIDATION ===

print("=== NULL CHECK ===")

df_silver_promotions.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_promotions.columns
]).show()


print("=== DUPLICATE PROMOTION ID CHECK ===")

df_silver_promotions.groupBy("promotion_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_promotions.count()
silver_count = df_silver_promotions.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Match:", bronze_count == silver_count)


print("=== INVALID PROMOTION ID CHECK ===")

print(
    "Invalid promotion IDs:",
    df_silver_promotions
    .filter(
        ~F.col("promotion_id").rlike("^PRM_[0-9]{3}$")
    )
    .count()
)


print("=== DISCOUNT BUSINESS RULE CHECK ===")

print(
    "Invalid discounts:",
    df_silver_promotions
    .filter(
        (F.col("discount_percentage") < 0) |
        (F.col("discount_percentage") > 1) |
        F.col("discount_percentage").isNull()
    )
    .count()
)


print("=== EMPTY STRING CHECK ===")

for c in ["promotion_id", "campaign_name", "marketing_channel"]:
    print(
        f"Empty {c}:",
        df_silver_promotions.filter(
            F.trim(F.col(c)) == ""
        ).count()
    )

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 23, Finished, Available, Finished, False)

=== NULL CHECK ===
+------------+-------------+-------------------+-----------------+------------+------------+--------------------+---------------+
|promotion_id|campaign_name|discount_percentage|marketing_channel|is_stackable|_source_file|_ingestion_timestamp|_ingestion_date|
+------------+-------------+-------------------+-----------------+------------+------------+--------------------+---------------+
|           0|            0|                  0|                0|           0|           0|                   0|              0|
+------------+-------------+-------------------+-----------------+------------+------------+--------------------+---------------+

=== DUPLICATE PROMOTION ID CHECK ===
+------------+-----+
|promotion_id|count|
+------------+-----+
+------------+-----+

=== ROW COUNT CHECK ===
Bronze rows: 30
Silver rows: 30
Match: True
=== INVALID PROMOTION ID CHECK ===
Invalid promotion IDs: 0
=== DISCOUNT BUSINESS RULE CHECK ===
Invalid discounts: 0
=== EMPTY STRING CHECK

In [ ]:
# === WRITE SILVER PROMOTIONS ===

df_silver_promotions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_promotions") 

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 24, Finished, Available, Finished, False)

In [ ]:
# === VERIFY SILVER PROMOTIONS ===

df_silver_promotions_check = spark.table(
    "2_silver.silver_promotions"
)

print("Rows:", df_silver_promotions_check.count())

df_silver_promotions_check.printSchema()

df_silver_promotions_check.show(10, truncate=False)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 25, Finished, Available, Finished, False)

Rows: 30
root
 |-- promotion_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- is_stackable: boolean (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+------------+----------------------+-------------------+-----------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|promotion_id|campaign_name         |discount_percentage|marketing_channel|is_stackable|_source_file                                                                                                                                                                        |_ingestion_timestamp     

### 15. Stores_Dataset

### Stores — Bronze to Silver

The Bronze stores table was loaded into a PySpark DataFrame and profiled for data quality issues.

### Load and Inspect
- Loaded `1_bronze.bronze_stores` into `df_bronze_stores`.
- Confirmed 50 rows and 9 columns.
- Inspected the schema and sample records.

### Profiling
- Checked NULL values across all columns.
- Checked duplicate `store_id` values.
- Validated store ID format.
- Reviewed store type distribution.
- Reviewed region distribution.
- Reviewed active/inactive store distribution.
- Checked for empty strings.
- Generated a statistical profile.

### Data Quality Findings
- No NULL values found.
- No duplicate `store_id` values found.
- No invalid store IDs found.
- No empty strings found.
- 4 store types identified.
- 4 regions identified.
- 48 active stores and 2 inactive stores.
- No structural corruption detected.

### Clean and Transform
- Trimmed store names.
- Trimmed store types.
- Trimmed city values.
- Trimmed region values.
- Standardized `store_id` to uppercase.
- No records were removed.

### Validation
- Bronze rows: 50.
- Silver rows: 50.
- Row count matched.
- No NULL values.
- No duplicate store IDs.
- No invalid store IDs.
- No empty strings.

### Next Step
- Write the validated DataFrame to `silver.silver_stores`.
- Complete deferred `store_id` foreign-key validation for Employees and Inventory Snapshots.


In [ ]:
# === LOAD AND INSPECT ===

df_bronze_stores = spark.table("1_bronze.bronze_stores")

df_bronze_stores.printSchema()

df_bronze_stores.show(10, truncate=False)

print(f"Rows: {df_bronze_stores.count()}")
print(f"Columns: {len(df_bronze_stores.columns)}")

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 26, Finished, Available, Finished, False)

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------+----------------------------+----------+--------------+------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|store_id|store_name                  |store_type|city          |region      |is_active|_source_file                                                                                                                                                                    |_ingestion_timestamp      |_ingestio

In [25]:
# === PROFILE ===

print("=== NULL CHECK ===")

df_bronze_stores.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_stores.columns
]).show()


print("=== DUPLICATE STORE ID CHECK ===")

df_bronze_stores.groupBy("store_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== STORE ID FORMAT CHECK ===")

print(
    "Invalid store IDs:",
    df_bronze_stores
    .filter(
        ~F.col("store_id").rlike("^STR_[0-9]{3}$")
    )
    .count()
)


print("=== STORE TYPE DISTRIBUTION ===")

df_bronze_stores.groupBy("store_type") \
    .count() \
    .orderBy("store_type") \
    .show()


print("=== REGION DISTRIBUTION ===")

df_bronze_stores.groupBy("region") \
    .count() \
    .orderBy("region") \
    .show()


print("=== ACTIVE STATUS DISTRIBUTION ===")

df_bronze_stores.groupBy("is_active") \
    .count() \
    .show()


print("=== EMPTY STRING CHECK ===")

for c in ["store_id", "store_name", "store_type", "city", "region"]:
    print(
        f"Empty {c}:",
        df_bronze_stores.filter(
            F.trim(F.col(c)) == ""
        ).count()
    )


print("=== STATISTICAL PROFILE ===")

df_bronze_stores.describe().show(truncate=False)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 27, Finished, Available, Finished, False)

=== NULL CHECK ===
+--------+----------+----------+----+------+---------+------------+--------------------+---------------+
|store_id|store_name|store_type|city|region|is_active|_source_file|_ingestion_timestamp|_ingestion_date|
+--------+----------+----------+----+------+---------+------------+--------------------+---------------+
|       0|         0|         0|   0|     0|        0|           0|                   0|              0|
+--------+----------+----------+----+------+---------+------------+--------------------+---------------+

=== DUPLICATE STORE ID CHECK ===
+--------+-----+
|store_id|count|
+--------+-----+
+--------+-----+

=== STORE ID FORMAT CHECK ===
Invalid store IDs: 0
=== STORE TYPE DISTRIBUTION ===
+----------+-----+
|store_type|count|
+----------+-----+
|   Express|   14|
|  Flagship|    5|
| Mall-Type|    5|
|  Standard|   26|
+----------+-----+

=== REGION DISTRIBUTION ===
+------------+-----+
|      region|count|
+------------+-----+
|Central-West|   12|
|   N

In [26]:
# === CLEAN AND TRANSFORM ===

df_silver_stores = (
    df_bronze_stores
    .withColumn(
        "store_id",
        F.upper(F.trim(F.col("store_id")))
    )
    .withColumn(
        "store_name",
        F.trim(F.col("store_name"))
    )
    .withColumn(
        "store_type",
        F.trim(F.col("store_type"))
    )
    .withColumn(
        "city",
        F.trim(F.col("city"))
    )
    .withColumn(
        "region",
        F.trim(F.col("region"))
    )
)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 28, Finished, Available, Finished, False)

In [27]:
# === SILVER STORES VALIDATION ===

print("=== NULL CHECK ===")

df_silver_stores.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_stores.columns
]).show()


print("=== DUPLICATE STORE ID CHECK ===")

df_silver_stores.groupBy("store_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_stores.count()
silver_count = df_silver_stores.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Match:", bronze_count == silver_count)


print("=== INVALID STORE ID CHECK ===")

print(
    "Invalid store IDs:",
    df_silver_stores
    .filter(
        ~F.col("store_id").rlike("^STR_[0-9]{3}$")
    )
    .count()
)


print("=== EMPTY STRING CHECK ===")

for c in ["store_id", "store_name", "store_type", "city", "region"]:
    print(
        f"Empty {c}:",
        df_silver_stores.filter(
            F.trim(F.col(c)) == ""
        ).count()
    )

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 29, Finished, Available, Finished, False)

=== NULL CHECK ===
+--------+----------+----------+----+------+---------+------------+--------------------+---------------+
|store_id|store_name|store_type|city|region|is_active|_source_file|_ingestion_timestamp|_ingestion_date|
+--------+----------+----------+----+------+---------+------------+--------------------+---------------+
|       0|         0|         0|   0|     0|        0|           0|                   0|              0|
+--------+----------+----------+----+------+---------+------------+--------------------+---------------+

=== DUPLICATE STORE ID CHECK ===
+--------+-----+
|store_id|count|
+--------+-----+
+--------+-----+

=== ROW COUNT CHECK ===
Bronze rows: 50
Silver rows: 50
Match: True
=== INVALID STORE ID CHECK ===
Invalid store IDs: 0
=== EMPTY STRING CHECK ===
Empty store_id: 0
Empty store_name: 0
Empty store_type: 0
Empty city: 0
Empty region: 0


In [ ]:
# === WRITE SILVER STORES ===

df_silver_stores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_stores") 

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 30, Finished, Available, Finished, False)

In [ ]:
# === VERIFY SILVER STORES ===

df_silver_stores_check = spark.table(
    "2_silver.silver_stores"
)

print("Rows:", df_silver_stores_check.count())

df_silver_stores_check.printSchema()

df_silver_stores_check.show(10, truncate=False)

StatementMeta(, a2658750-7e09-48ba-ae65-914b76489075, 31, Finished, Available, Finished, False)

Rows: 50
root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+--------+----------------------------+----------+--------------+------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+
|store_id|store_name                  |store_type|city          |region      |is_active|_source_file                                                                                                                                                                    |_ingestion_timestamp      |

### 16. Suppliers_Dataset

In [1]:
from pyspark.sql import functions as F

StatementMeta(, 12069c9a-fd81-430b-beea-e3093002cf2f, 3, Finished, Available, Finished, False)

### Suppliers — Bronze to Silver

The Bronze suppliers table was loaded into a PySpark DataFrame and profiled for data quality and business-rule consistency.

### Load and Inspect
- Loaded `1_bronze.bronze_suppliers` into `df_bronze_suppliers`.
- Confirmed 250 rows and 8 columns.
- Inspected the schema and sample records.

### Profiling
- Checked NULL values across all columns.
- Checked duplicate `supplier_id` values.
- Validated supplier ID format.
- Reviewed supplier tier distribution.
- Reviewed primary category distribution.
- Reviewed country distribution.
- Checked for empty strings.
- Generated a statistical profile.

### Data Quality Findings
- No NULL values found.
- No duplicate `supplier_id` values found.
- No invalid supplier IDs found.
- No empty strings found.
- Identified 6 primary supplier categories.
- Identified 4 supplier countries.
- Identified 3 supplier tiers.
- Detected leading whitespace in `Tier 3 - Standard`.

### Clean and Transform
- Standardized `supplier_id` to uppercase and trimmed whitespace.
- Trimmed supplier names.
- Trimmed primary categories.
- Trimmed supplier tiers.
- Trimmed country values.
- Removed leading/trailing whitespace from supplier tier values.
- No records were removed.

### Validation
- Bronze rows: 250.
- Silver rows: 250.
- Row count matched.
- No NULL values.
- No duplicate supplier IDs.
- No invalid supplier IDs.
- No empty strings.
- No invalid supplier tiers.
- Confirmed zero supplier tier values contained leading or trailing whitespace.

### Output
- Written to `silver.silver_suppliers`.
- Final Silver row count: 250.
- Schema verified after writing.


In [ ]:
# === LOAD AND INSPECT ===

df_bronze_suppliers = spark.table("1_bronze.bronze_suppliers")

df_bronze_suppliers.printSchema()

df_bronze_suppliers.show(10, truncate=False)

print(f"Rows: {df_bronze_suppliers.count()}")
print(f"Columns: {len(df_bronze_suppliers.columns)}")

StatementMeta(, 12069c9a-fd81-430b-beea-e3093002cf2f, 4, Finished, Available, Finished, False)

root
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- supplier_tier: string (nullable = true)
 |-- country: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+-----------+---------------------+----------------+------------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+---------------+
|supplier_id|supplier_name        |primary_category|supplier_tier     |country      |_source_file                                                                                                                                                                        |_ingestion_timestamp     |_ingestion_date|
+-----------

In [3]:
# === PROFILE ===

print("=== NULL CHECK ===")

df_bronze_suppliers.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_bronze_suppliers.columns
]).show()


print("=== DUPLICATE SUPPLIER ID CHECK ===")

df_bronze_suppliers.groupBy("supplier_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== SUPPLIER ID FORMAT CHECK ===")

print(
    "Invalid supplier IDs:",
    df_bronze_suppliers
    .filter(
        ~F.col("supplier_id").rlike("^SUP_[0-9]{4}$")
    )
    .count()
)


print("=== SUPPLIER TIER DISTRIBUTION ===")

df_bronze_suppliers.groupBy("supplier_tier") \
    .count() \
    .orderBy("supplier_tier") \
    .show()


print("=== PRIMARY CATEGORY DISTRIBUTION ===")

df_bronze_suppliers.groupBy("primary_category") \
    .count() \
    .orderBy("primary_category") \
    .show()


print("=== COUNTRY DISTRIBUTION ===")

df_bronze_suppliers.groupBy("country") \
    .count() \
    .orderBy("country") \
    .show()


print("=== EMPTY STRING CHECK ===")

for c in [
    "supplier_id",
    "supplier_name",
    "primary_category",
    "supplier_tier",
    "country"
]:
    print(
        f"Empty {c}:",
        df_bronze_suppliers.filter(
            F.trim(F.col(c)) == ""
        ).count()
    )


print("=== STATISTICAL PROFILE ===")

df_bronze_suppliers.describe().show(truncate=False)

StatementMeta(, 12069c9a-fd81-430b-beea-e3093002cf2f, 5, Finished, Available, Finished, False)

=== NULL CHECK ===
+-----------+-------------+----------------+-------------+-------+------------+--------------------+---------------+
|supplier_id|supplier_name|primary_category|supplier_tier|country|_source_file|_ingestion_timestamp|_ingestion_date|
+-----------+-------------+----------------+-------------+-------+------------+--------------------+---------------+
|          0|            0|               0|            0|      0|           0|                   0|              0|
+-----------+-------------+----------------+-------------+-------+------------+--------------------+---------------+

=== DUPLICATE SUPPLIER ID CHECK ===
+-----------+-----+
|supplier_id|count|
+-----------+-----+
+-----------+-----+

=== SUPPLIER ID FORMAT CHECK ===
Invalid supplier IDs: 0
=== SUPPLIER TIER DISTRIBUTION ===
+------------------+-----+
|     supplier_tier|count|
+------------------+-----+
|Tier 1 - Strategic|   39|
|Tier 2 - Preferred|  115|
| Tier 3 - Standard|   96|
+------------------+----

In [4]:
# === CLEAN AND TRANSFORM ===

df_silver_suppliers = (
    df_bronze_suppliers
    .withColumn(
        "supplier_id",
        F.upper(F.trim(F.col("supplier_id")))
    )
    .withColumn(
        "supplier_name",
        F.trim(F.col("supplier_name"))
    )
    .withColumn(
        "primary_category",
        F.trim(F.col("primary_category"))
    )
    .withColumn(
        "supplier_tier",
        F.trim(F.col("supplier_tier"))
    )
    .withColumn(
        "country",
        F.trim(F.col("country"))
    )
)

StatementMeta(, 12069c9a-fd81-430b-beea-e3093002cf2f, 6, Finished, Available, Finished, False)

In [5]:
# === SILVER SUPPLIERS VALIDATION ===

print("=== NULL CHECK ===")

df_silver_suppliers.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver_suppliers.columns
]).show()


print("=== DUPLICATE SUPPLIER ID CHECK ===")

df_silver_suppliers.groupBy("supplier_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


print("=== ROW COUNT CHECK ===")

bronze_count = df_bronze_suppliers.count()
silver_count = df_silver_suppliers.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Match:", bronze_count == silver_count)


print("=== INVALID SUPPLIER ID CHECK ===")

print(
    "Invalid supplier IDs:",
    df_silver_suppliers
    .filter(
        ~F.col("supplier_id").rlike("^SUP_[0-9]{4}$")
    )
    .count()
)


print("=== EMPTY STRING CHECK ===")

for c in [
    "supplier_id",
    "supplier_name",
    "primary_category",
    "supplier_tier",
    "country"
]:
    print(
        f"Empty {c}:",
        df_silver_suppliers.filter(
            F.trim(F.col(c)) == ""
        ).count()
    )


print("=== SUPPLIER TIER FORMAT CHECK ===")

print(
    "Invalid supplier tiers:",
    df_silver_suppliers
    .filter(
        ~F.col("supplier_tier").isin(
            "Tier 1 - Strategic",
            "Tier 2 - Preferred",
            "Tier 3 - Standard"
        )
    )
    .count()
)

StatementMeta(, 12069c9a-fd81-430b-beea-e3093002cf2f, 7, Finished, Available, Finished, False)

=== NULL CHECK ===
+-----------+-------------+----------------+-------------+-------+------------+--------------------+---------------+
|supplier_id|supplier_name|primary_category|supplier_tier|country|_source_file|_ingestion_timestamp|_ingestion_date|
+-----------+-------------+----------------+-------------+-------+------------+--------------------+---------------+
|          0|            0|               0|            0|      0|           0|                   0|              0|
+-----------+-------------+----------------+-------------+-------+------------+--------------------+---------------+

=== DUPLICATE SUPPLIER ID CHECK ===
+-----------+-----+
|supplier_id|count|
+-----------+-----+
+-----------+-----+

=== ROW COUNT CHECK ===
Bronze rows: 250
Silver rows: 250
Match: True
=== INVALID SUPPLIER ID CHECK ===
Invalid supplier IDs: 0
=== EMPTY STRING CHECK ===
Empty supplier_id: 0
Empty supplier_name: 0
Empty primary_category: 0
Empty supplier_tier: 0
Empty country: 0
=== SUPPLIER 

In [7]:
# === SUPPLIER TIER CLEANING CHECK ===

df_silver_suppliers.select(
    "supplier_tier"
).distinct().show(truncate=False)

StatementMeta(, 12069c9a-fd81-430b-beea-e3093002cf2f, 9, Finished, Available, Finished, False)

+------------------+
|supplier_tier     |
+------------------+
|Tier 1 - Strategic|
|Tier 3 - Standard |
|Tier 2 - Preferred|
+------------------+



In [6]:
print(
    "Tiers with leading/trailing whitespace:",
    df_silver_suppliers
    .filter(
        F.col("supplier_tier") != F.trim(F.col("supplier_tier"))
    )
    .count()
)

StatementMeta(, 12069c9a-fd81-430b-beea-e3093002cf2f, 8, Finished, Available, Finished, False)

Tiers with leading/trailing whitespace: 0


In [ ]:
# === WRITE SILVER SUPPLIERS ===

df_silver_suppliers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("2_silver.silver_suppliers")

StatementMeta(, 12069c9a-fd81-430b-beea-e3093002cf2f, 10, Finished, Available, Finished, False)

In [ ]:
# === VERIFY SILVER SUPPLIERS ===

df_silver_suppliers_check = spark.table(
    "2_silver.silver_suppliers"
)

print("Rows:", df_silver_suppliers_check.count())

df_silver_suppliers_check.printSchema()

df_silver_suppliers_check.show(10, truncate=False)

StatementMeta(, 12069c9a-fd81-430b-beea-e3093002cf2f, 11, Finished, Available, Finished, False)

Rows: 250
root
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- supplier_tier: string (nullable = true)
 |-- country: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)

+-----------+---------------------+----------------+------------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+---------------+
|supplier_id|supplier_name        |primary_category|supplier_tier     |country      |_source_file                                                                                                                                                                        |_ingestion_timestamp     |_ingestion_date|
+-

In [ ]:
# Gold Preparation — Orders & Order Items Validation

from pyspark.sql.functions import col

# 1. Load Silver tables
orders_df = spark.table("2_silver.silver_orders")
order_items_df = spark.table("2_silver.silver_order_items")

orders_count = orders_df.count()
order_items_count = order_items_df.count()

print("=== SILVER TABLE ROW COUNTS ===")
print(f"silver_orders: {orders_count:,}")
print(f"silver_order_items: {order_items_count:,}")


# 2. Validate order_id uniqueness
duplicate_orders = (
    orders_df
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_orders_count = duplicate_orders.count()

print("\n=== ORDER PRIMARY KEY VALIDATION ===")
print(f"Duplicate order_id count: {duplicate_orders_count:,}")

if duplicate_orders_count == 0:
    print("PASS: order_id is unique in silver_orders.")
else:
    print("FAIL: Duplicate order_id values detected.")

duplicate_orders.show(10, truncate=False)


# 3. Validate order-item grain
duplicate_order_items = (
    order_items_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_order_items_count = duplicate_order_items.count()

print("\n=== ORDER ITEM GRAIN VALIDATION ===")
print(
    f"Duplicate (order_id, order_item_id) count: "
    f"{duplicate_order_items_count:,}"
)

if duplicate_order_items_count == 0:
    print(
        "PASS: (order_id, order_item_id) is unique "
        "in silver_order_items."
    )
else:
    print("FAIL: Duplicate order-item keys detected.")

duplicate_order_items.show(10, truncate=False)


# 4. Validate referential integrity
orphan_order_items = (
    order_items_df
    .select("order_id")
    .distinct()
    .join(
        orders_df.select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
)

orphan_order_items_count = orphan_order_items.count()

print("\n=== ORDER REFERENTIAL INTEGRITY VALIDATION ===")
print(f"Orphan order_id count: {orphan_order_items_count:,}")

if orphan_order_items_count == 0:
    print(
        "PASS: Every order_id in silver_order_items "
        "exists in silver_orders."
    )
else:
    print("FAIL: Orphan order_id values detected.")

orphan_order_items.show(10, truncate=False)


# 5. Validate NULL keys
null_order_id_orders = orders_df.filter(
    col("order_id").isNull()
).count()

null_order_id_items = order_items_df.filter(
    col("order_id").isNull()
).count()

null_order_item_id = order_items_df.filter(
    col("order_item_id").isNull()
).count()

print("\n=== NULL KEY VALIDATION ===")
print(f"Null order_id in silver_orders: {null_order_id_orders:,}")
print(f"Null order_id in silver_order_items: {null_order_id_items:,}")
print(f"Null order_item_id in silver_order_items: {null_order_item_id:,}")

if (
    null_order_id_orders == 0
    and null_order_id_items == 0
    and null_order_item_id == 0
):
    print("PASS: No NULL values found in required key columns.")
else:
    print("FAIL: NULL values detected in required key columns.")


# 6. Inspect order-item distribution
order_item_distribution = (
    order_items_df
    .groupBy("order_id")
    .count()
)

print("\n=== ORDER ITEM DISTRIBUTION ===")
print("Orders with the highest number of order items:")

order_item_distribution \
    .orderBy(col("count").desc()) \
    .show(10, truncate=False)


# 7. Validation summary
all_validations_passed = (
    duplicate_orders_count == 0
    and duplicate_order_items_count == 0
    and orphan_order_items_count == 0
    and null_order_id_orders == 0
    and null_order_id_items == 0
    and null_order_item_id == 0
)

print("\n=== GOLD PREPARATION VALIDATION SUMMARY ===")

if all_validations_passed:
    print("STATUS: PASS")
    print(
        "Silver orders and order_items are ready "
        "for fact_sales modelling."
    )
else:
    print("STATUS: FAIL")
    print(
        "Silver orders/order_items require investigation "
        "before creating fact_sales."
    )
    

StatementMeta(, ef81d557-20cf-4800-8c88-c7b081cddf0d, 4, Finished, Available, Finished, False)

=== SILVER TABLE ROW COUNTS ===
silver_orders: 9,946
silver_order_items: 11,335

=== ORDER PRIMARY KEY VALIDATION ===
Duplicate order_id count: 0
PASS: order_id is unique in silver_orders.
+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+


=== ORDER ITEM GRAIN VALIDATION ===
Duplicate (order_id, order_item_id) count: 0
PASS: (order_id, order_item_id) is unique in silver_order_items.
+--------+-------------+-----+
|order_id|order_item_id|count|
+--------+-------------+-----+
+--------+-------------+-----+


=== ORDER REFERENTIAL INTEGRITY VALIDATION ===
Orphan order_id count: 0
PASS: Every order_id in silver_order_items exists in silver_orders.
+--------+
|order_id|
+--------+
+--------+


=== NULL KEY VALIDATION ===
Null order_id in silver_orders: 0
Null order_id in silver_order_items: 0
Null order_item_id in silver_order_items: 0
PASS: No NULL values found in required key columns.

=== ORDER ITEM DISTRIBUTION ===
Orders with the highest number of order items:
+-----

In [3]:
# Gold — fact_sales preview

from pyspark.sql.functions import col, to_date

fact_sales_preview = (
    orders_df.alias("o")
    .join(
        order_items_df.alias("oi"),
        col("o.order_id") == col("oi.order_id"),
        "inner"
    )
    .select(
        col("o.order_id"),
        col("oi.order_item_id"),
        col("o.customer_id"),
        col("oi.product_id"),
        col("oi.seller_id"),
        to_date(
            col("o.order_purchase_timestamp")
        ).alias("order_date"),
        col("o.order_status"),
        col("oi.price"),
        col("oi.freight_value")
    )
)

fact_sales_preview.show(20, truncate=False)

print("fact_sales preview rows:", fact_sales_preview.count())

StatementMeta(, ef81d557-20cf-4800-8c88-c7b081cddf0d, 5, Finished, Available, Finished, False)

+--------------------------------+-------------+--------------------------------+--------------------------------+--------------------------------+----------+------------+------+-------------+
|order_id                        |order_item_id|customer_id                     |product_id                      |seller_id                       |order_date|order_status|price |freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+--------------------------------+----------+------------+------+-------------+
|00063b381e2406b52ad429470734ebd5|1            |6a899e55865de6549a58d2c6845e5604|f177554ea93259a5b282f24e33f65ab6|8602a61d680a10a82cceeeda0d99ea3d|2018-07-27|delivered   |45.00 |12.98        |
|000e63d38ae8c00bbcb5a30573b99628|1            |98884e672c5ba85f4394f2044e1a3eab|553e0e7590d3116a072507a3635d2877|1c129092bf23f28a5930387c980c0dfc|2018-03-23|delivered   |47.90 |8.88         |
|00137e170939bba5a3134e2386413108|1